In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars seaborn

import polars as pl

In [3]:
!nvidia-smi

Sat Apr 25 17:37:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D1:00.0 Off |                  Off |
| 35%   65C    P2            189W /  230W |    9941MiB /  24564MiB |     82%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
import pandas as pd
import os

BASE_PATH = 'vindr_mammogram'
meta_path = os.path.join(BASE_PATH, 'metadata.csv')

device = pd.read_csv(meta_path)
device.head()

,SOP Instance UID,Series Instance UID,SOP Instance UID.1,Patient's Age,View Position,Image Laterality,Photometric Interpretation,Rows,Columns,Imager Pixel Spacing,...,Pixel Padding Value,Pixel Padding Range Limit,Window Center,Window Width,Rescale Intercept,Rescale Slope,Rescale Type,Window Center & Width Explanation,Manufacturer,Manufacturer's Model Name
0,d8125545210c08e1b1793a5af6458ee2,b36517b9cbbcfd286a7ae04f643af97a,d8125545210c08e1b1793a5af6458ee2,053Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1662,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
1,290c658f4e75a3f83ec78a847414297c,b36517b9cbbcfd286a7ae04f643af97a,290c658f4e75a3f83ec78a847414297c,053Y,MLO,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1664,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
2,cd0fc7bc53ac632a11643ac4cc91002a,b36517b9cbbcfd286a7ae04f643af97a,cd0fc7bc53ac632a11643ac4cc91002a,053Y,CC,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1600,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
3,71638b1e853799f227492bfb08a01491,b36517b9cbbcfd286a7ae04f643af97a,71638b1e853799f227492bfb08a01491,053Y,MLO,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1654,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
4,dd9ce3288c0773e006a294188aadba8e,d931832a0815df082c085b6e09d20aac,dd9ce3288c0773e006a294188aadba8e,042Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1580,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration


In [6]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [7]:
df['cc_finding_categories'].value_counts().sort_index()

cc_finding_categories
['Architectural Distortion', 'Mass']                                                                   1
['Architectural Distortion']                                                                          40
['Asymmetry', 'Mass']                                                                                  1
['Asymmetry']                                                                                         20
['Focal Asymmetry']                                                                                  107
['Global Asymmetry']                                                                                  11
['Mass']                                                                                             443
['Nipple Retraction', 'Mass']                                                                          1
['Nipple Retraction', 'Skin Thickening', 'Mass']                                                       1
['Nipple Retraction']            

In [8]:
import pandas as pd

device_info = device[['SOP Instance UID', "Manufacturer's Model Name"]].copy()
device_info.columns = ['image_id', 'device_model']

df = df.merge(device_info, left_on='cc_image_id', right_on='image_id', how='left')
df = df.drop('image_id', axis=1)

device_mapping = {
    'Mammomat Inspiration': 0,
    'Planmed Nuance': 1,
    'GIOTTO CLASS': 2,
    'GIOTTO IMAGE 3DL': 2
}

df['device_id'] = df['device_model'].map(device_mapping)

print(df['device_model'].value_counts())
print(df['device_id'].value_counts())
print(df.shape)

device_model
Mammomat Inspiration    7621
Planmed Nuance          1898
GIOTTO CLASS             314
GIOTTO IMAGE 3DL         166
Name: count, dtype: int64
device_id
0    7621
1    1898
2     480
Name: count, dtype: int64
(9999, 26)


In [9]:
device_mapping = {
    'DENSITY A': 0,
    'DENSITY B': 1,
    'DENSITY C': 2,
    'DENSITY D': 3
}

df['density'] = df['cc_breast_density'].map(device_mapping)

In [10]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 0
        else:
            return 1 if max_birads == 3 else 4
    
    if has_structural:
        return 4
    
    if has_mass and has_calc:
        return 3
    
    if has_mass:
        return 2
    
    if has_calc:
        return 1
    
    if has_lymph:
        return 3
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 4
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 4

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    9032
1     132
2     490
3      66
4      90
Name: count, dtype: int64

In [11]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [12]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['label'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,5,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [13]:
inbreast_df['label'].value_counts()

label
1    98
0    30
4    28
3    20
2    11
Name: count, dtype: int64

In [14]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['label'] = df['cc_breast_birads'].apply(birads_to_label)

In [15]:
df['birads']=df['label'].copy()

In [16]:
df['density'].value_counts()

density
2    7486
3    1335
1     942
0      47
Name: count, dtype: int64

In [17]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [18]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [19]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [20]:
train_df.shape

(7057, 36)

In [21]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [22]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import gc
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, precision_score, recall_score,
    roc_curve, auc as sklearn_auc, classification_report
)
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}\n")

NUM_FINDINGS = 6
IMG_SIZE = 512
BATCH_SIZE = 8

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]

FINDING_MAP = {
    0: 'Negative BI-RADS 1',
    1: 'Negative BI-RADS 2',
    2: 'Mass',
    3: 'Calcification',
    4: 'Structural/Complex',
    5: 'Lymphadenopathy',
}

DEVICE_MAP = {
    0: 'Mammomat Inspiration',
    1: 'Planmed Nuance',
    2: 'GIOTTO',
}

DENSITY_MAP = {
    0: 'Density A (Fatty)',
    1: 'Density B (Scattered)',
    2: 'Density C (Heterogeneous)',
    3: 'Density D (Dense)',
}

BINARY_CLASS_NAMES = ["Negative (BI-RADS 1)", "Positive (BI-RADS 2-5)"]

# Model checkpoint paths
MODEL_CHECKPOINTS = {
    'efficientnet_b3': '/workspace/Thesis_updated_results/binary_512_dual_proto/efficientnet_b3/best_model.pth',
    'convnext_base': '/workspace/Thesis_updated_results/binary_512_dual_proto/convnext_base/best_model.pth',
}

OUTPUT_BASE_DIR = '/workspace/Thesis_updated_results/binary_analysis_results'


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class MammoDataset(Dataset):
    def __init__(self, df, transform, dataset_name='default'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
        if 'label' in df.columns:
            counts = df['label'].value_counts().sort_index().to_dict()
            print(f"{dataset_name.upper()} distribution: {counts}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = None
        for col in ['merged_image_path', 'image_path', 'img_path', 'filepath']:
            if col in row:
                img_path = row[col]
                break
        
        if img_path is None:
            raise ValueError(f"No image path found in row {idx}")
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='gray')
        
        image = self.transform(image)
        
        birads = int(row['label']) if 'label' in row else 0
        binary = 0 if birads == 0 else 1
        density = int(row['density']) if 'density' in row else 0
        device_id = int(row['device_id']) if 'device_id' in row else 0
        
        finding_vec = np.zeros(NUM_FINDINGS, dtype=np.float32)
        for f_id, col_name in enumerate(FINDING_COLS):
            if col_name in row:
                try:
                    finding_vec[f_id] = float(row[col_name])
                except:
                    finding_vec[f_id] = 0.0
        
        return (
            img_path, image,
            torch.tensor(birads, dtype=torch.long),
            torch.tensor(binary, dtype=torch.long),
            torch.tensor(density, dtype=torch.long),
            torch.tensor(device_id, dtype=torch.long),
            torch.tensor(finding_vec, dtype=torch.float32),
        )


# ─────────────────────────────────────────────────────────────────────────────
# 2. MODEL COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


class FindingAwareProtoModel(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128, dropout=0.4):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone
        self.is_cnn = True

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.binary_head = nn.Sequential(
            nn.Linear(self.num_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]
        
        if self.is_cnn and f.ndim == 4:
            f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
        
        return f

    def forward(self, x):
        feat = self._extract(x)
        binary_logit = self.binary_head(feat)
        return binary_logit


class DualProtoNet(nn.Module):
    def __init__(self, backbone_name, backbone_fn, backbone_weights,
                 emb_dim=128, dropout=0.4, num_findings=NUM_FINDINGS):
        super().__init__()
        self.num_findings = num_findings
        
        backbone = backbone_fn(weights=backbone_weights)
        self.encoder = FindingAwareProtoModel(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
        )

        self.register_buffer('proto_finding', 
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1))
        self.register_buffer('proto_birads', 
            F.normalize(torch.randn(2, emb_dim), dim=-1))

    def forward_encoder(self, x):
        return self.encoder(x)


# ─────────────────────────────────────────────────────────────────────────────
# 3. UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def to_py(v):
    if isinstance(v, torch.Tensor):
        return v.item()
    if isinstance(v, np.generic):
        return v.item()
    return v


def compute_metrics(y_true, y_pred, y_proba=None):
    """Compute binary classification metrics"""
    if len(y_true) == 0 or len(y_pred) == 0:
        return {
            'Accuracy': np.nan,
            'Precision': np.nan,
            'Recall': np.nan,
            'Macro-F1': np.nan,
            'AUC': np.nan,
        }
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    auc = np.nan
    if y_proba is not None and len(np.unique(y_true)) > 1:
        try:
            auc = roc_auc_score(y_true, y_proba[:, 1])
        except:
            auc = np.nan
    
    return {
        'Accuracy': round(float(acc), 4),
        'Precision': round(float(prec), 4),
        'Recall': round(float(recall), 4),
        'Macro-F1': round(float(macro_f1), 4),
        'AUC': round(float(auc), 4) if not np.isnan(auc) else '—'
    }


@torch.no_grad()
def run_inference(model, loader):
    """Run inference and return results dataframe"""
    model.eval()
    
    paths_l, labels_l, preds_l, probs_l = [], [], [], []
    density_l, device_ids_l, finding_vecs_l = [], [], []
    birads_l = []
    
    for batch in tqdm(loader, desc="Inference", leave=False):
        paths, images, birads, binary, densities, device_ids, finding_vecs = batch
        images = images.to(DEVICE)
        
        binary_logit = model.forward_encoder(images)
        probs = F.softmax(binary_logit, dim=1)
        preds = binary_logit.argmax(1)
        
        paths_l.extend(list(paths))
        labels_l.extend([to_py(v) for v in binary])
        preds_l.extend([to_py(v) for v in preds])
        probs_l.extend([p.cpu().numpy() for p in probs])
        density_l.extend([to_py(v) for v in densities])
        device_ids_l.extend([to_py(v) for v in device_ids])
        birads_l.extend([to_py(v) for v in birads])
        finding_vecs_l.extend([fv.cpu().numpy() for fv in finding_vecs])
    
    df = pd.DataFrame({
        'path': paths_l,
        'birads_label': birads_l,
        'binary_label': labels_l,
        'pred': preds_l,
        'density': density_l,
        'device_id': device_ids_l,
    })
    
    probs_arr = np.stack(probs_l, axis=0)
    for c in range(2):
        df[f'prob_cls{c}'] = probs_arr[:, c]
    
    finding_matrix = np.array(finding_vecs_l)
    for f_id in range(NUM_FINDINGS):
        df[f'finding_{f_id}'] = finding_matrix[:, f_id]
    
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 4. ANALYSIS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def analyze_device(df_res, output_dir):
    """Analyze performance by device"""
    results = []
    
    for dev_id in sorted(df_res['device_id'].unique()):
        mask = df_res['device_id'] == dev_id
        subset = df_res[mask]
        
        y_true = subset['binary_label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(2)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        cm = confusion_matrix(y_true, y_pred)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
        else:
            tn, fp, fn, tp = 0, 0, 0, 0
        
        row = {
            'Device': DEVICE_MAP.get(int(dev_id), f'Device {dev_id}'),
            'N': len(subset),
            'TN': int(tn),
            'TP': int(tp),
            'FN': int(fn),
            'FP': int(fp),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'binary_per_device.csv'), index=False)
    
    print(f"\nPER-DEVICE ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_density(df_res, output_dir):
    """Analyze performance by density (VinDr only)"""
    results = []
    
    for dens_id in sorted(df_res['density'].unique()):
        mask = df_res['density'] == dens_id
        subset = df_res[mask]
        
        y_true = subset['binary_label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(2)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        row = {
            'Density': DENSITY_MAP.get(int(dens_id), f'Density {dens_id}'),
            'N': len(subset),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'binary_per_density.csv'), index=False)
    
    print(f"\nPER-DENSITY ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_finding(df_res, output_dir):
    """Analyze performance by finding"""
    results = []
    
    for f_id in range(NUM_FINDINGS):
        col = f'finding_{f_id}'
        mask = df_res[col] > 0.5
        
        if mask.sum() < 1:
            continue
        
        subset = df_res[mask]
        y_true = subset['binary_label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(2)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        # Binary-specific counts
        cm = confusion_matrix(y_true, y_pred)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
        else:
            tn, fp, fn, tp = 0, 0, 0, 0
        
        row = {
            'Finding': FINDING_MAP.get(f_id, f'Finding {f_id}'),
            'N': len(subset),
            'TN_Correct': int(tn),
            'TP_Correct': int(tp),
            'FN_Miss': int(fn),
            'FP_Miss': int(fp),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'binary_per_finding.csv'), index=False)
    
    print(f"\nPER-FINDING ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_vindr(df_res, model_name, output_dir):
    """Complete VinDr analysis with all stratifications"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['binary_label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(2)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"VINDR BINARY ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    # Overall
    metrics = compute_metrics(y_true, y_pred, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    
    overall_row = {
        'Category': 'Overall',
        'N': len(df_res),
        'TN': int(tn),
        'TP': int(tp),
        'FN': int(fn),
        'FP': int(fp),
        **metrics
    }
    
    print(f"\nOVERALL:\n{pd.DataFrame([overall_row]).to_string(index=False)}\n")
    
    # Save overall
    pd.DataFrame([overall_row]).to_csv(os.path.join(output_dir, 'binary_overall.csv'), index=False)
    
    # Device analysis
    analyze_device(df_res, output_dir)
    
    # Density analysis
    analyze_density(df_res, output_dir)
    
    # Finding analysis
    analyze_finding(df_res, output_dir)


def analyze_inbreast(df_res, model_name, output_dir):
    """INbreast analysis (summary only, no stratification)"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['binary_label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(2)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"INBREAST BINARY ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    metrics = compute_metrics(y_true, y_pred, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    
    result_row = {
        'Dataset': 'INbreast',
        'N': len(df_res),
        'TN': int(tn),
        'TP': int(tp),
        'FN': int(fn),
        'FP': int(fp),
        **metrics
    }
    
    print(f"\n{pd.DataFrame([result_row]).to_string(index=False)}\n")
    
    pd.DataFrame([result_row]).to_csv(os.path.join(output_dir, 'binary_overall.csv'), index=False)


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_analysis(test_df, inbreast_df_input):
    """Run complete binary analysis for both models"""
    
    print(f"\n{'#'*100}")
    print(f"# BINARY CLASSIFICATION - COMPREHENSIVE ANALYSIS")
    print(f"# EfficientNet-B3 & ConvNeXt-Base | VinDr & INbreast")
    print(f"{'#'*100}\n")
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # All results
    all_summaries = []
    
    # For each model
    for model_name, checkpoint_path in MODEL_CHECKPOINTS.items():
        if not os.path.exists(checkpoint_path):
            print(f"Skipping {model_name}: checkpoint not found\n")
            continue
        
        print(f"\n{'*'*100}")
        print(f"* MODEL: {model_name.upper()}")
        print(f"{'*'*100}\n")
        
        # Load model
        if 'efficientnet' in model_name:
            backbone_fn = models.efficientnet_b3
            backbone_weights = models.EfficientNet_B3_Weights.DEFAULT
        else:
            backbone_fn = models.convnext_base
            backbone_weights = models.ConvNeXt_Base_Weights.DEFAULT
        
        model = DualProtoNet(
            backbone_name=model_name,
            backbone_fn=backbone_fn,
            backbone_weights=backbone_weights,
            emb_dim=128, dropout=0.4,
            num_findings=NUM_FINDINGS
        ).to(DEVICE)
        
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        state_dict = ckpt['model_state_dict']
        
        # Convert 5-class proto_birads to 2-class if needed
        if 'proto_birads' in state_dict and state_dict['proto_birads'].shape[0] == 5:
            print("Converting 5-class BI-RADS prototype to 2-class binary...")
            proto_birads_5 = state_dict['proto_birads']
            proto_birads_2 = torch.zeros(2, 128, dtype=proto_birads_5.dtype)
            proto_birads_2[0] = proto_birads_5[0]  # Class 0 (Negative)
            # Merge classes 1-4 into class 1 (Positive)
            proto_birads_2[1] = (proto_birads_5[1] + proto_birads_5[2] + proto_birads_5[3] + proto_birads_5[4]).mean(dim=0)
            proto_birads_2 = F.normalize(proto_birads_2, dim=1)
            state_dict['proto_birads'] = proto_birads_2
            state_dict['proto_birads_updates'] = torch.tensor([5, 5], dtype=torch.long)
        
        model.load_state_dict(state_dict, strict=False)
        print(f"Loaded: Epoch {ckpt.get('epoch', 0)+1}, Best F1: {ckpt.get('best_macro_f1', 'N/A')}\n")
        
        # Loaders
        loaders = {}
        if test_df is not None:
            ds = MammoDataset(test_df, transform, 'vindr')
            loaders['vindr'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        if inbreast_df_input is not None:
            ds = MammoDataset(inbreast_df_input, transform, 'inbreast')
            loaders['inbreast'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        # For each split
        for split_name, loader in loaders.items():
            print(f"\nProcessing {split_name.upper()}...")
            df_res = run_inference(model, loader)
            
            # Output directory
            output_dir = os.path.join(OUTPUT_BASE_DIR, f'{model_name}_512', split_name)
            
            # Get summary metrics
            y_true = df_res['binary_label'].values
            y_pred = df_res['pred'].values
            y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(2)], axis=1)
            metrics = compute_metrics(y_true, y_pred, y_proba)
            
            summary = {
                'Model': model_name.upper(),
                'Split': split_name.upper(),
                'N': len(df_res),
                **metrics
            }
            all_summaries.append(summary)
            
            # Analysis
            if split_name == 'vindr':
                analyze_vindr(df_res, model_name, output_dir)
            else:
                analyze_inbreast(df_res, model_name, output_dir)
        
        del model
        torch.cuda.empty_cache()
        gc.collect()
    
    # Master summary
    if all_summaries:
        df_master = pd.DataFrame(all_summaries)
        master_csv = os.path.join(OUTPUT_BASE_DIR, 'binary_master_summary.csv')
        os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
        df_master.to_csv(master_csv, index=False)
        
        print(f"\n\n{'='*100}")
        print(f"MASTER SUMMARY TABLE - BINARY CLASSIFICATION")
        print(f"{'='*100}\n{df_master.to_string(index=False)}\n")
        print(f"Saved to: {master_csv}")
    
    print(f"\n{'='*100}")
    print(f"BINARY ANALYSIS COMPLETE!")
    print(f"Results saved to: {OUTPUT_BASE_DIR}/")
    print(f"{'='*100}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 6. RUN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    # Replace with your actual dataframes
    run_analysis(test_df, inbreast_df)


Device: cuda


####################################################################################################
# BINARY CLASSIFICATION - COMPREHENSIVE ANALYSIS
# EfficientNet-B3 & ConvNeXt-Base | VinDr & INbreast
####################################################################################################


****************************************************************************************************
* MODEL: EFFICIENTNET_B3
****************************************************************************************************

Converting 5-class BI-RADS prototype to 2-class binary...
Loaded: Epoch 33, Best F1: 0.8116207951070336

VINDR distribution: {0: 1341, 1: 467, 2: 67, 3: 69, 4: 22}
INBREAST distribution: {0: 30, 1: 98, 2: 11, 3: 20, 4: 28}

Processing VINDR...



VINDR BINARY ANALYSIS - EFFICIENTNET_B3

OVERALL:
Category    N   TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
 Overall 1966 1199 405 220 142    0.8159     0.7404   0.648      0.78 0.8325


PER-DEVICE ANALYSIS:
              Device    N  TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
Mammomat Inspiration 1496 931 295 167 103    0.8195     0.7412  0.6385    0.7797 0.8274
      Planmed Nuance  393 233  76  49  35    0.7863     0.6847  0.6080    0.7457 0.8166
              GIOTTO   77  35  34   4   4    0.8961     0.8947  0.8947    0.8961 0.9345


PER-DENSITY ANALYSIS:
                  Density    N  Accuracy  Precision  Recall  Macro-F1     AUC
        Density A (Fatty)   10    0.8000     0.0000  0.0000    0.4444       —
    Density B (Scattered)  188    0.8138     0.7719  0.6667    0.7886  0.8224
Density C (Heterogeneous) 1501    0.8188     0.7599  0.6369    0.7822  0.8357
        Density D (Dense)  267    0.8015     0.6429  0.7013    0.7644  0.8303


PER-


INBREAST BINARY ANALYSIS - EFFICIENTNET_B3

 Dataset   N  TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
INbreast 187  27 127  30   3    0.8235     0.9769  0.8089    0.7529 0.9157


****************************************************************************************************
* MODEL: CONVNEXT_BASE
****************************************************************************************************

Converting 5-class BI-RADS prototype to 2-class binary...
Loaded: Epoch 36, Best F1: 0.8272657351621586

VINDR distribution: {0: 1341, 1: 467, 2: 67, 3: 69, 4: 22}
INBREAST distribution: {0: 30, 1: 98, 2: 11, 3: 20, 4: 28}

Processing VINDR...



VINDR BINARY ANALYSIS - CONVNEXT_BASE

OVERALL:
Category    N   TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
 Overall 1966 1212 424 201 129    0.8321     0.7667  0.6784       0.8 0.8574


PER-DEVICE ANALYSIS:
              Device    N  TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
Mammomat Inspiration 1496 937 309 153  97    0.8329     0.7611  0.6688    0.7971 0.8582
      Planmed Nuance  393 241  83  42  27    0.8244     0.7545  0.6640    0.7906 0.8327
              GIOTTO   77  34  32   6   5    0.8571     0.8649  0.8421    0.8570 0.9305


PER-DENSITY ANALYSIS:
                  Density    N  Accuracy  Precision  Recall  Macro-F1     AUC
        Density A (Fatty)   10    0.9000     0.0000  0.0000    0.4737       —
    Density B (Scattered)  188    0.8298     0.8542  0.6212    0.7986  0.8507
Density C (Heterogeneous) 1501    0.8348     0.7799  0.6763    0.8032  0.8556
        Density D (Dense)  267    0.8165     0.6628  0.7403    0.7837  0.8779


PER-FI


INBREAST BINARY ANALYSIS - CONVNEXT_BASE

 Dataset   N  TN  TP  FN  FP  Accuracy  Precision  Recall  Macro-F1    AUC
INbreast 187  27 128  29   3    0.8289     0.9771  0.8153    0.7584 0.9136



MASTER SUMMARY TABLE - BINARY CLASSIFICATION
          Model    Split    N  Accuracy  Precision  Recall  Macro-F1    AUC
EFFICIENTNET_B3    VINDR 1966    0.8159     0.7404  0.6480    0.7800 0.8325
EFFICIENTNET_B3 INBREAST  187    0.8235     0.9769  0.8089    0.7529 0.9157
  CONVNEXT_BASE    VINDR 1966    0.8321     0.7667  0.6784    0.8000 0.8574
  CONVNEXT_BASE INBREAST  187    0.8289     0.9771  0.8153    0.7584 0.9136

Saved to: /workspace/Thesis_updated_results/binary_analysis_results/binary_master_summary.csv

BINARY ANALYSIS COMPLETE!
Results saved to: /workspace/Thesis_updated_results/binary_analysis_results/



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import gc
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, precision_score, recall_score,
    roc_curve, auc as sklearn_auc, classification_report
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}\n")

NUM_CLASSES = 5
NUM_FINDINGS = 6
IMG_SIZE = 512
BATCH_SIZE = 8

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]

FINDING_MAP = {
    0: 'Negative BI-RADS 1',
    1: 'Negative BI-RADS 2',
    2: 'Mass',
    3: 'Calcification',
    4: 'Structural/Complex',
    5: 'Lymphadenopathy',
}

DEVICE_MAP = {
    0: 'Mammomat Inspiration',
    1: 'Planmed Nuance',
    2: 'GIOTTO',
}

DENSITY_MAP = {
    0: 'Density A (Fatty)',
    1: 'Density B (Scattered)',
    2: 'Density C (Heterogeneous)',
    3: 'Density D (Dense)',
}

BIRADS_CLASS_NAMES = ["BI-RADS 0", "BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4"]

# Model checkpoint paths
MODEL_CHECKPOINTS = {
    'efficientnet_b3': '/workspace/Thesis_updated_results/birads_512_dual_proto/efficientnet_b3/best_model.pth',
    'convnext_base': '/workspace/Thesis_updated_results/birads_512_dual_proto/convnext_base/best_model.pth',
}

OUTPUT_BASE_DIR = '/workspace/Thesis_updated_results/birads_analysis_results'


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class MammoDataset(Dataset):
    def __init__(self, df, transform, dataset_name='default'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
        if 'label' in df.columns:
            counts = df['label'].value_counts().sort_index().to_dict()
            print(f"{dataset_name.upper()} BI-RADS distribution: {counts}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = None
        for col in ['merged_image_path', 'image_path', 'img_path', 'filepath']:
            if col in row:
                img_path = row[col]
                break
        
        if img_path is None:
            raise ValueError(f"No image path found in row {idx}")
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='gray')
        
        image = self.transform(image)
        
        birads = int(row['label']) if 'label' in row else 0
        density = int(row['density']) if 'density' in row else 0
        device_id = int(row['device_id']) if 'device_id' in row else 0
        
        finding_vec = np.zeros(NUM_FINDINGS, dtype=np.float32)
        for f_id, col_name in enumerate(FINDING_COLS):
            if col_name in row:
                try:
                    finding_vec[f_id] = float(row[col_name])
                except:
                    finding_vec[f_id] = 0.0
        
        return (
            img_path, image,
            torch.tensor(birads, dtype=torch.long),
            torch.tensor(density, dtype=torch.long),
            torch.tensor(device_id, dtype=torch.long),
            torch.tensor(finding_vec, dtype=torch.float32),
        )


# ─────────────────────────────────────────────────────────────────────────────
# 2. MODEL COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


class FindingAwareProtoModel(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128, dropout=0.4, num_classes=5):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone
        self.num_classes = num_classes
        self.is_cnn = True

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.cls_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes - 1),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]
        
        if self.is_cnn and f.ndim == 4:
            f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
        
        return f

    def forward(self, x):
        feat = self._extract(x)
        logits = self.cls_head(feat)
        return logits


class DualProtoNet(nn.Module):
    def __init__(self, backbone_name, backbone_fn, backbone_weights,
                 emb_dim=128, dropout=0.4, num_classes=5, num_findings=NUM_FINDINGS):
        super().__init__()
        self.num_classes = num_classes
        self.num_findings = num_findings
        
        backbone = backbone_fn(weights=backbone_weights)
        self.encoder = FindingAwareProtoModel(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        self.register_buffer('proto_finding', 
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1))
        self.register_buffer('proto_birads', 
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1))

    def forward_encoder(self, x):
        return self.encoder(x)


# ─────────────────────────────────────────────────────────────────────────────
# 3. UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def to_py(v):
    if isinstance(v, torch.Tensor):
        return v.item()
    if isinstance(v, np.generic):
        return v.item()
    return v


def ordinal_logits_to_label(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)


def ordinal_logits_to_probs(logits):
    q = torch.sigmoid(logits)
    B = q.size(0)
    K = q.size(1) + 1
    probs = torch.zeros(B, K, device=logits.device, dtype=logits.dtype)
    
    probs[:, 0] = 1.0 - q[:, 0]
    for c in range(1, K - 1):
        probs[:, c] = q[:, c - 1] - q[:, c]
    probs[:, K - 1] = q[:, K - 2]
    
    return probs.clamp(min=1e-8, max=1.0)


def compute_auc_ovr(y_true, y_proba):
    """Compute One-vs-Rest AUC"""
    try:
        aucs = []
        for class_id in range(NUM_CLASSES):
            y_bin = (y_true == class_id).astype(int)
            pos = y_bin.sum()
            neg = (1 - y_bin).sum()
            
            if pos > 0 and neg > 0:
                try:
                    auc = roc_auc_score(y_bin, y_proba[:, class_id])
                    aucs.append(auc)
                except:
                    pass
        
        return np.mean(aucs) if aucs else np.nan
    except:
        return np.nan


def compute_metrics(y_true, y_pred, y_proba=None):
    """Compute BI-RADS 5-class metrics"""
    if len(y_true) == 0:
        return {
            'Accuracy': np.nan,
            'Precision': np.nan,
            'Recall': np.nan,
            'Macro-F1': np.nan,
            'AUC': np.nan,
        }
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    auc = compute_auc_ovr(y_true, y_proba) if y_proba is not None else np.nan
    
    return {
        'Accuracy': round(float(acc), 4),
        'Precision': round(float(prec), 4),
        'Recall': round(float(recall), 4),
        'Macro-F1': round(float(macro_f1), 4),
        'AUC': round(float(auc), 4) if not np.isnan(auc) else '—'
    }


@torch.no_grad()
def run_inference(model, loader):
    """Run inference and return results dataframe"""
    model.eval()
    
    paths_l, labels_l, preds_l, probs_l = [], [], [], []
    density_l, device_ids_l, finding_vecs_l = [], [], []
    
    for batch in tqdm(loader, desc="Inference", leave=False):
        paths, images, birads, densities, device_ids, finding_vecs = batch
        images = images.to(DEVICE)
        
        logits = model.forward_encoder(images)
        probs = ordinal_logits_to_probs(logits)
        preds = ordinal_logits_to_label(logits)
        
        paths_l.extend(list(paths))
        labels_l.extend([to_py(v) for v in birads])
        preds_l.extend([to_py(v) for v in preds])
        probs_l.extend([p.cpu().numpy() for p in probs])
        density_l.extend([to_py(v) for v in densities])
        device_ids_l.extend([to_py(v) for v in device_ids])
        finding_vecs_l.extend([fv.cpu().numpy() for fv in finding_vecs])
    
    df = pd.DataFrame({
        'path': paths_l,
        'label': labels_l,
        'pred': preds_l,
        'density': density_l,
        'device_id': device_ids_l,
    })
    
    probs_arr = np.stack(probs_l, axis=0)
    for c in range(NUM_CLASSES):
        df[f'prob_cls{c}'] = probs_arr[:, c]
    
    finding_matrix = np.array(finding_vecs_l)
    for f_id in range(NUM_FINDINGS):
        df[f'finding_{f_id}'] = finding_matrix[:, f_id]
    
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 4. ANALYSIS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def analyze_device(df_res, output_dir):
    """Analyze performance by device"""
    results = []
    
    for dev_id in sorted(df_res['device_id'].unique()):
        mask = df_res['device_id'] == dev_id
        subset = df_res[mask]
        
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        row = {
            'Device': DEVICE_MAP.get(int(dev_id), f'Device {dev_id}'),
            'N': len(subset),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_device.csv'), index=False)
    
    print(f"\nPER-DEVICE ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_density(df_res, output_dir):
    """Analyze performance by density (VinDr only)"""
    results = []
    
    for dens_id in sorted(df_res['density'].unique()):
        mask = df_res['density'] == dens_id
        subset = df_res[mask]
        
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        row = {
            'Density': DENSITY_MAP.get(int(dens_id), f'Density {dens_id}'),
            'N': len(subset),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_density.csv'), index=False)
    
    print(f"\nPER-DENSITY ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_finding(df_res, output_dir):
    """Analyze performance by finding with BI-RADS level breakdown"""
    results = []
    
    for f_id in range(NUM_FINDINGS):
        col = f'finding_{f_id}'
        mask = df_res[col] > 0.5
        
        if mask.sum() < 1:
            continue
        
        subset = df_res[mask]
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        # BI-RADS Level Breakdown
        # BI-RADS 0-1 (Low Risk)
        low_risk_correct = ((y_true <= 1) & (y_pred <= 1)).sum()
        low_risk_incorrect = ((y_true <= 1) & (y_pred > 1)).sum()
        
        # BI-RADS 2 (Benign with Callback)
        birads2_correct = ((y_true == 2) & (y_pred == 2)).sum()
        birads2_incorrect = ((y_true == 2) & (y_pred != 2)).sum()
        
        # BI-RADS 3-4 (High Risk - Suspicious/Malignant)
        high_risk_correct = ((y_true >= 3) & (y_pred >= 3)).sum()
        high_risk_incorrect = ((y_true >= 3) & (y_pred < 3)).sum()
        
        # Overall counts
        exact_correct = (y_true == y_pred).sum()
        exact_incorrect = (y_true != y_pred).sum()
        
        row = {
            'Finding': FINDING_MAP.get(f_id, f'Finding {f_id}'),
            'N': len(subset),
            'LowRisk_0-1_Correct': int(low_risk_correct),
            'LowRisk_0-1_Incorrect': int(low_risk_incorrect),
            'BIRADS2_Correct': int(birads2_correct),
            'BIRADS2_Incorrect': int(birads2_incorrect),
            'HighRisk_3-4_Correct': int(high_risk_correct),
            'HighRisk_3-4_Incorrect': int(high_risk_incorrect),
            'Exact_Match': int(exact_correct),
            'Exact_Mismatch': int(exact_incorrect),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_finding.csv'), index=False)
    
    print(f"\nPER-FINDING ANALYSIS (BI-RADS Level Breakdown):")
    print(f"Legend: LowRisk=BI-RADS 0-1 | BIRADS2=BI-RADS 2 | HighRisk=BI-RADS 3-4")
    print(f"{df_result.to_string(index=False)}\n")
    return df_result


def analyze_vindr(df_res, model_name, output_dir):
    """Complete VinDr analysis with all stratifications"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"VINDR BI-RADS ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    # Overall
    metrics = compute_metrics(y_true, y_pred, y_proba)
    
    overall_row = {
        'Category': 'Overall',
        'N': len(df_res),
        **metrics
    }
    
    print(f"\nOVERALL:\n{pd.DataFrame([overall_row]).to_string(index=False)}\n")
    
    # Save overall
    pd.DataFrame([overall_row]).to_csv(os.path.join(output_dir, 'birads_overall.csv'), index=False)
    
    # Device analysis
    analyze_device(df_res, output_dir)
    
    # Density analysis
    analyze_density(df_res, output_dir)
    
    # Finding analysis
    analyze_finding(df_res, output_dir)


def analyze_inbreast(df_res, model_name, output_dir):
    """INbreast analysis (summary only, no stratification)"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"INBREAST BI-RADS ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    metrics = compute_metrics(y_true, y_pred, y_proba)
    
    result_row = {
        'Dataset': 'INbreast',
        'N': len(df_res),
        **metrics
    }
    
    print(f"\n{pd.DataFrame([result_row]).to_string(index=False)}\n")
    
    pd.DataFrame([result_row]).to_csv(os.path.join(output_dir, 'birads_overall.csv'), index=False)


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_analysis(test_df, inbreast_df_input):
    """Run complete BI-RADS analysis for both models"""
    
    print(f"\n{'#'*100}")
    print(f"# BI-RADS 5-CLASS CLASSIFICATION - COMPREHENSIVE ANALYSIS")
    print(f"# EfficientNet-B3 & ConvNeXt-Base | VinDr & INbreast")
    print(f"{'#'*100}\n")
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # All results
    all_summaries = []
    
    # For each model
    for model_name, checkpoint_path in MODEL_CHECKPOINTS.items():
        if not os.path.exists(checkpoint_path):
            print(f"Skipping {model_name}: checkpoint not found\n")
            continue
        
        print(f"\n{'*'*100}")
        print(f"* MODEL: {model_name.upper()}")
        print(f"{'*'*100}\n")
        
        # Load model
        if 'efficientnet' in model_name:
            backbone_fn = models.efficientnet_b3
            backbone_weights = models.EfficientNet_B3_Weights.DEFAULT
        else:
            backbone_fn = models.convnext_base
            backbone_weights = models.ConvNeXt_Base_Weights.DEFAULT
        
        model = DualProtoNet(
            backbone_name=model_name,
            backbone_fn=backbone_fn,
            backbone_weights=backbone_weights,
            emb_dim=128, dropout=0.4,
            num_classes=NUM_CLASSES, num_findings=NUM_FINDINGS
        ).to(DEVICE)
        
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        print(f"Loaded: Epoch {ckpt.get('epoch', 0)+1}, Best F1: {ckpt.get('best_macro_f1', 'N/A')}\n")
        
        # Loaders
        loaders = {}
        if test_df is not None:
            ds = MammoDataset(test_df, transform, 'vindr')
            loaders['vindr'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        if inbreast_df_input is not None:
            ds = MammoDataset(inbreast_df_input, transform, 'inbreast')
            loaders['inbreast'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        # For each split
        for split_name, loader in loaders.items():
            print(f"\nProcessing {split_name.upper()}...")
            df_res = run_inference(model, loader)
            
            # Output directory
            output_dir = os.path.join(OUTPUT_BASE_DIR, f'{model_name}_512', split_name)
            
            # Get summary metrics
            y_true = df_res['label'].values
            y_pred = df_res['pred'].values
            y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
            metrics = compute_metrics(y_true, y_pred, y_proba)
            
            summary = {
                'Model': model_name.upper(),
                'Split': split_name.upper(),
                'N': len(df_res),
                **metrics
            }
            all_summaries.append(summary)
            
            # Analysis
            if split_name == 'vindr':
                analyze_vindr(df_res, model_name, output_dir)
            else:
                analyze_inbreast(df_res, model_name, output_dir)
        
        del model
        torch.cuda.empty_cache()
        gc.collect()
    
    # Master summary
    if all_summaries:
        df_master = pd.DataFrame(all_summaries)
        master_csv = os.path.join(OUTPUT_BASE_DIR, 'birads_master_summary.csv')
        os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
        df_master.to_csv(master_csv, index=False)
        
        print(f"\n\n{'='*100}")
        print(f"MASTER SUMMARY TABLE - BI-RADS 5-CLASS CLASSIFICATION")
        print(f"{'='*100}\n{df_master.to_string(index=False)}\n")
        print(f"Saved to: {master_csv}")
    
    print(f"\n{'='*100}")
    print(f"BI-RADS ANALYSIS COMPLETE!")
    print(f"Results saved to: {OUTPUT_BASE_DIR}/")
    print(f"{'='*100}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 6. RUN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    # Replace with your actual dataframes
    run_analysis(test_df, inbreast_df)

In [ ]:
sssssssss

In [ ]:
hhhhhhhhhhh

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import gc
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, precision_score, recall_score,
    roc_curve, auc as sklearn_auc, classification_report
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}\n")

NUM_CLASSES = 5
NUM_FINDINGS = 6
IMG_SIZE = 512
BATCH_SIZE = 8

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]

FINDING_MAP = {
    0: 'Negative BI-RADS 1',
    1: 'Negative BI-RADS 2',
    2: 'Mass',
    3: 'Calcification',
    4: 'Structural/Complex',
    5: 'Lymphadenopathy',
}

DEVICE_MAP = {
    0: 'Mammomat Inspiration',
    1: 'Planmed Nuance',
    2: 'GIOTTO',
}

DENSITY_MAP = {
    0: 'Density A (Fatty)',
    1: 'Density B (Scattered)',
    2: 'Density C (Heterogeneous)',
    3: 'Density D (Dense)',
}

BIRADS_CLASS_NAMES = ["BI-RADS 0", "BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4"]

# Model checkpoint paths
MODEL_CHECKPOINTS = {
    'efficientnet_b3': '/workspace/Thesis_updated_results/birads_512_dual_proto/efficientnet_b3/best_model.pth',
    'convnext_base': '/workspace/Thesis_updated_results/birads_512_dual_proto/convnext_base/best_model.pth',
}

OUTPUT_BASE_DIR = '/workspace/Thesis_updated_results/birads_analysis_results'


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class MammoDataset(Dataset):
    def __init__(self, df, transform, dataset_name='default'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
        if 'label' in df.columns:
            counts = df['label'].value_counts().sort_index().to_dict()
            print(f"{dataset_name.upper()} BI-RADS distribution: {counts}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = None
        for col in ['merged_image_path', 'image_path', 'img_path', 'filepath']:
            if col in row:
                img_path = row[col]
                break
        
        if img_path is None:
            raise ValueError(f"No image path found in row {idx}")
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='gray')
        
        image = self.transform(image)
        
        birads = int(row['label']) if 'label' in row else 0
        density = int(row['density']) if 'density' in row else 0
        device_id = int(row['device_id']) if 'device_id' in row else 0
        
        finding_vec = np.zeros(NUM_FINDINGS, dtype=np.float32)
        for f_id, col_name in enumerate(FINDING_COLS):
            if col_name in row:
                try:
                    finding_vec[f_id] = float(row[col_name])
                except:
                    finding_vec[f_id] = 0.0
        
        return (
            img_path, image,
            torch.tensor(birads, dtype=torch.long),
            torch.tensor(density, dtype=torch.long),
            torch.tensor(device_id, dtype=torch.long),
            torch.tensor(finding_vec, dtype=torch.float32),
        )


# ─────────────────────────────────────────────────────────────────────────────
# 2. MODEL COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


class FindingAwareProtoModel(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128, dropout=0.4, num_classes=5):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone
        self.num_classes = num_classes
        self.is_cnn = True

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.cls_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes - 1),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]
        
        if self.is_cnn and f.ndim == 4:
            f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
        
        return f

    def forward(self, x):
        feat = self._extract(x)
        logits = self.cls_head(feat)
        return logits


class DualProtoNet(nn.Module):
    def __init__(self, backbone_name, backbone_fn, backbone_weights,
                 emb_dim=128, dropout=0.4, num_classes=5, num_findings=NUM_FINDINGS):
        super().__init__()
        self.num_classes = num_classes
        self.num_findings = num_findings
        
        backbone = backbone_fn(weights=backbone_weights)
        self.encoder = FindingAwareProtoModel(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        self.register_buffer('proto_finding', 
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1))
        self.register_buffer('proto_birads', 
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1))

    def forward_encoder(self, x):
        return self.encoder(x)


# ─────────────────────────────────────────────────────────────────────────────
# 3. UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def to_py(v):
    if isinstance(v, torch.Tensor):
        return v.item()
    if isinstance(v, np.generic):
        return v.item()
    return v


def ordinal_logits_to_label(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)


def ordinal_logits_to_probs(logits):
    q = torch.sigmoid(logits)
    B = q.size(0)
    K = q.size(1) + 1
    probs = torch.zeros(B, K, device=logits.device, dtype=logits.dtype)
    
    probs[:, 0] = 1.0 - q[:, 0]
    for c in range(1, K - 1):
        probs[:, c] = q[:, c - 1] - q[:, c]
    probs[:, K - 1] = q[:, K - 2]
    
    return probs.clamp(min=1e-8, max=1.0)


def compute_auc_ovr(y_true, y_proba):
    """Compute One-vs-Rest AUC"""
    try:
        aucs = []
        for class_id in range(NUM_CLASSES):
            y_bin = (y_true == class_id).astype(int)
            pos = y_bin.sum()
            neg = (1 - y_bin).sum()
            
            if pos > 0 and neg > 0:
                try:
                    auc = roc_auc_score(y_bin, y_proba[:, class_id])
                    aucs.append(auc)
                except:
                    pass
        
        return np.mean(aucs) if aucs else np.nan
    except:
        return np.nan


def compute_metrics(y_true, y_pred, y_proba=None):
    """Compute BI-RADS 5-class metrics"""
    if len(y_true) == 0:
        return {
            'Accuracy': np.nan,
            'Precision': np.nan,
            'Recall': np.nan,
            'Macro-F1': np.nan,
            'AUC': np.nan,
        }
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    auc = compute_auc_ovr(y_true, y_proba) if y_proba is not None else np.nan
    
    return {
        'Accuracy': round(float(acc), 4),
        'Precision': round(float(prec), 4),
        'Recall': round(float(recall), 4),
        'Macro-F1': round(float(macro_f1), 4),
        'AUC': round(float(auc), 4) if not np.isnan(auc) else '—'
    }


@torch.no_grad()
def run_inference(model, loader):
    """Run inference and return results dataframe"""
    model.eval()
    
    paths_l, labels_l, preds_l, probs_l = [], [], [], []
    density_l, device_ids_l, finding_vecs_l = [], [], []
    
    for batch in tqdm(loader, desc="Inference", leave=False):
        paths, images, birads, densities, device_ids, finding_vecs = batch
        images = images.to(DEVICE)
        
        logits = model.forward_encoder(images)
        probs = ordinal_logits_to_probs(logits)
        preds = ordinal_logits_to_label(logits)
        
        paths_l.extend(list(paths))
        labels_l.extend([to_py(v) for v in birads])
        preds_l.extend([to_py(v) for v in preds])
        probs_l.extend([p.cpu().numpy() for p in probs])
        density_l.extend([to_py(v) for v in densities])
        device_ids_l.extend([to_py(v) for v in device_ids])
        finding_vecs_l.extend([fv.cpu().numpy() for fv in finding_vecs])
    
    df = pd.DataFrame({
        'path': paths_l,
        'label': labels_l,
        'pred': preds_l,
        'density': density_l,
        'device_id': device_ids_l,
    })
    
    probs_arr = np.stack(probs_l, axis=0)
    for c in range(NUM_CLASSES):
        df[f'prob_cls{c}'] = probs_arr[:, c]
    
    finding_matrix = np.array(finding_vecs_l)
    for f_id in range(NUM_FINDINGS):
        df[f'finding_{f_id}'] = finding_matrix[:, f_id]
    
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 4. ANALYSIS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def analyze_device(df_res, output_dir):
    """Analyze performance by device"""
    results = []
    
    for dev_id in sorted(df_res['device_id'].unique()):
        mask = df_res['device_id'] == dev_id
        subset = df_res[mask]
        
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        row = {
            'Device': DEVICE_MAP.get(int(dev_id), f'Device {dev_id}'),
            'N': len(subset),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_device.csv'), index=False)
    
    print(f"\nPER-DEVICE ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_density(df_res, output_dir):
    """Analyze performance by density (VinDr only)"""
    results = []
    
    for dens_id in sorted(df_res['density'].unique()):
        mask = df_res['density'] == dens_id
        subset = df_res[mask]
        
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        row = {
            'Density': DENSITY_MAP.get(int(dens_id), f'Density {dens_id}'),
            'N': len(subset),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_density.csv'), index=False)
    
    print(f"\nPER-DENSITY ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_finding(df_res, output_dir):
    """Analyze performance by finding"""
    results = []
    
    for f_id in range(NUM_FINDINGS):
        col = f'finding_{f_id}'
        mask = df_res[col] > 0.5
        
        if mask.sum() < 1:
            continue
        
        subset = df_res[mask]
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        y_proba = np.stack([subset[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
        
        metrics = compute_metrics(y_true, y_pred, y_proba)
        
        # Count correct vs incorrect
        correct = (y_true == y_pred).sum()
        incorrect = (y_true != y_pred).sum()
        
        row = {
            'Finding': FINDING_MAP.get(f_id, f'Finding {f_id}'),
            'N': len(subset),
            'Correct': int(correct),
            'Incorrect': int(incorrect),
            **metrics
        }
        
        results.append(row)
    
    df_result = pd.DataFrame(results)
    os.makedirs(output_dir, exist_ok=True)
    df_result.to_csv(os.path.join(output_dir, 'birads_per_finding.csv'), index=False)
    
    print(f"\nPER-FINDING ANALYSIS:\n{df_result.to_string(index=False)}\n")
    return df_result


def analyze_vindr(df_res, model_name, output_dir):
    """Complete VinDr analysis with all stratifications"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"VINDR BI-RADS ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    # Overall
    metrics = compute_metrics(y_true, y_pred, y_proba)
    
    overall_row = {
        'Category': 'Overall',
        'N': len(df_res),
        **metrics
    }
    
    print(f"\nOVERALL:\n{pd.DataFrame([overall_row]).to_string(index=False)}\n")
    
    # Save overall
    pd.DataFrame([overall_row]).to_csv(os.path.join(output_dir, 'birads_overall.csv'), index=False)
    
    # Device analysis
    analyze_device(df_res, output_dir)
    
    # Density analysis
    analyze_density(df_res, output_dir)
    
    # Finding analysis
    analyze_finding(df_res, output_dir)


def analyze_inbreast(df_res, model_name, output_dir):
    """INbreast analysis (summary only, no stratification)"""
    os.makedirs(output_dir, exist_ok=True)
    
    y_true = df_res['label'].values
    y_pred = df_res['pred'].values
    y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
    
    print(f"\n{'='*100}")
    print(f"INBREAST BI-RADS ANALYSIS - {model_name.upper()}")
    print(f"{'='*100}")
    
    metrics = compute_metrics(y_true, y_pred, y_proba)
    
    result_row = {
        'Dataset': 'INbreast',
        'N': len(df_res),
        **metrics
    }
    
    print(f"\n{pd.DataFrame([result_row]).to_string(index=False)}\n")
    
    pd.DataFrame([result_row]).to_csv(os.path.join(output_dir, 'birads_overall.csv'), index=False)


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN EVALUATION LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_analysis(test_df, inbreast_df_input):
    """Run complete BI-RADS analysis for both models"""
    
    print(f"\n{'#'*100}")
    print(f"# BI-RADS 5-CLASS CLASSIFICATION - COMPREHENSIVE ANALYSIS")
    print(f"# EfficientNet-B3 & ConvNeXt-Base | VinDr & INbreast")
    print(f"{'#'*100}\n")
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # All results
    all_summaries = []
    
    # For each model
    for model_name, checkpoint_path in MODEL_CHECKPOINTS.items():
        if not os.path.exists(checkpoint_path):
            print(f"Skipping {model_name}: checkpoint not found\n")
            continue
        
        print(f"\n{'*'*100}")
        print(f"* MODEL: {model_name.upper()}")
        print(f"{'*'*100}\n")
        
        # Load model
        if 'efficientnet' in model_name:
            backbone_fn = models.efficientnet_b3
            backbone_weights = models.EfficientNet_B3_Weights.DEFAULT
        else:
            backbone_fn = models.convnext_base
            backbone_weights = models.ConvNeXt_Base_Weights.DEFAULT
        
        model = DualProtoNet(
            backbone_name=model_name,
            backbone_fn=backbone_fn,
            backbone_weights=backbone_weights,
            emb_dim=128, dropout=0.4,
            num_classes=NUM_CLASSES, num_findings=NUM_FINDINGS
        ).to(DEVICE)
        
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        print(f"Loaded: Epoch {ckpt.get('epoch', 0)+1}, Best F1: {ckpt.get('best_macro_f1', 'N/A')}\n")
        
        # Loaders
        loaders = {}
        if test_df is not None:
            ds = MammoDataset(test_df, transform, 'vindr')
            loaders['vindr'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        if inbreast_df_input is not None:
            ds = MammoDataset(inbreast_df_input, transform, 'inbreast')
            loaders['inbreast'] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
        
        # For each split
        for split_name, loader in loaders.items():
            print(f"\nProcessing {split_name.upper()}...")
            df_res = run_inference(model, loader)
            
            # Output directory
            output_dir = os.path.join(OUTPUT_BASE_DIR, f'{model_name}_512', split_name)
            
            # Get summary metrics
            y_true = df_res['label'].values
            y_pred = df_res['pred'].values
            y_proba = np.stack([df_res[f'prob_cls{c}'].values for c in range(NUM_CLASSES)], axis=1)
            metrics = compute_metrics(y_true, y_pred, y_proba)
            
            summary = {
                'Model': model_name.upper(),
                'Split': split_name.upper(),
                'N': len(df_res),
                **metrics
            }
            all_summaries.append(summary)
            
            # Analysis
            if split_name == 'vindr':
                analyze_vindr(df_res, model_name, output_dir)
            else:
                analyze_inbreast(df_res, model_name, output_dir)
        
        del model
        torch.cuda.empty_cache()
        gc.collect()
    
    # Master summary
    if all_summaries:
        df_master = pd.DataFrame(all_summaries)
        master_csv = os.path.join(OUTPUT_BASE_DIR, 'birads_master_summary.csv')
        os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
        df_master.to_csv(master_csv, index=False)
        
        print(f"\n\n{'='*100}")
        print(f"MASTER SUMMARY TABLE - BI-RADS 5-CLASS CLASSIFICATION")
        print(f"{'='*100}\n{df_master.to_string(index=False)}\n")
        print(f"Saved to: {master_csv}")
    
    print(f"\n{'='*100}")
    print(f"BI-RADS ANALYSIS COMPLETE!")
    print(f"Results saved to: {OUTPUT_BASE_DIR}/")
    print(f"{'='*100}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 6. RUN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    # Replace with your actual dataframes
    run_analysis(test_df, inbreast_df)


Device: cuda


####################################################################################################
# BI-RADS 5-CLASS CLASSIFICATION - COMPREHENSIVE ANALYSIS
# EfficientNet-B3 & ConvNeXt-Base | VinDr & INbreast
####################################################################################################


****************************************************************************************************
* MODEL: EFFICIENTNET_B3
****************************************************************************************************

Loaded: Epoch 49, Best F1: 0.6287666605495149

VINDR BI-RADS distribution: {0: 1341, 1: 467, 2: 67, 3: 69, 4: 22}
INBREAST BI-RADS distribution: {0: 30, 1: 98, 2: 11, 3: 20, 4: 28}

Processing VINDR...



VINDR BI-RADS ANALYSIS - EFFICIENTNET_B3

OVERALL:
Category    N  Accuracy  Precision  Recall  Macro-F1   AUC
 Overall 1966     0.763      0.613  0.4862     0.532 0.801


PER-DEVICE ANALYSIS:
              Device    N  Accuracy  Precision  Recall  Macro-F1    AUC
Mammomat Inspiration 1496    0.7741     0.6779  0.4770    0.5412 0.7828
      Planmed Nuance  393    0.7252     0.4166  0.3424    0.3633 0.7750
              GIOTTO   77    0.7403     0.5889  0.5563    0.5703 0.8320


PER-DENSITY ANALYSIS:
                  Density    N  Accuracy  Precision  Recall  Macro-F1     AUC
        Density A (Fatty)   10    1.0000     1.0000  1.0000    1.0000       —
    Density B (Scattered)  188    0.7766     0.7261  0.6709    0.6892  0.9009
Density C (Heterogeneous) 1501    0.7515     0.5790  0.4593    0.5000  0.7928
        Density D (Dense)  267    0.8090     0.5038  0.4629    0.4750  0.7956


PER-FINDING ANALYSIS:
           Finding    N  Correct  Incorrect  Accuracy  Precision  Recall  Macro-F


INBREAST BI-RADS ANALYSIS - EFFICIENTNET_B3

 Dataset   N  Accuracy  Precision  Recall  Macro-F1    AUC
INbreast 187    0.6364     0.5419  0.5277    0.5138 0.8051


****************************************************************************************************
* MODEL: CONVNEXT_BASE
****************************************************************************************************

Loaded: Epoch 38, Best F1: 0.6245621277753666

VINDR BI-RADS distribution: {0: 1341, 1: 467, 2: 67, 3: 69, 4: 22}
INBREAST BI-RADS distribution: {0: 30, 1: 98, 2: 11, 3: 20, 4: 28}

Processing VINDR...



VINDR BI-RADS ANALYSIS - CONVNEXT_BASE

OVERALL:
Category    N  Accuracy  Precision  Recall  Macro-F1    AUC
 Overall 1966     0.796     0.6006  0.5165    0.5466 0.8103


PER-DEVICE ANALYSIS:
              Device    N  Accuracy  Precision  Recall  Macro-F1    AUC
Mammomat Inspiration 1496    0.8015     0.6541  0.5177    0.5673 0.8092
      Planmed Nuance  393    0.7863     0.5154  0.4455    0.4727 0.7464
              GIOTTO   77    0.7403     0.5436  0.5347    0.5296 0.9168


PER-DENSITY ANALYSIS:
                  Density    N  Accuracy  Precision  Recall  Macro-F1     AUC
        Density A (Fatty)   10    0.9000     0.5000  0.4500    0.4737       —
    Density B (Scattered)  188    0.7819     0.7209  0.6646    0.6759  0.9118
Density C (Heterogeneous) 1501    0.7995     0.5680  0.5075    0.5235  0.8133
        Density D (Dense)  267    0.7828     0.4993  0.4669    0.4778  0.7725


PER-FINDING ANALYSIS:
           Finding    N  Correct  Incorrect  Accuracy  Precision  Recall  Macro-F


INBREAST BI-RADS ANALYSIS - CONVNEXT_BASE

 Dataset   N  Accuracy  Precision  Recall  Macro-F1    AUC
INbreast 187    0.6257     0.5698  0.5603    0.5358 0.7927



MASTER SUMMARY TABLE - BI-RADS 5-CLASS CLASSIFICATION
          Model    Split    N  Accuracy  Precision  Recall  Macro-F1    AUC
EFFICIENTNET_B3    VINDR 1966    0.7630     0.6130  0.4862    0.5320 0.8010
EFFICIENTNET_B3 INBREAST  187    0.6364     0.5419  0.5277    0.5138 0.8051
  CONVNEXT_BASE    VINDR 1966    0.7960     0.6006  0.5165    0.5466 0.8103
  CONVNEXT_BASE INBREAST  187    0.6257     0.5698  0.5603    0.5358 0.7927

Saved to: /workspace/Thesis_updated_results/birads_analysis_results/birads_master_summary.csv

BI-RADS ANALYSIS COMPLETE!
Results saved to: /workspace/Thesis_updated_results/birads_analysis_results/



In [ ]:
ssssssssss

In [ ]:
dddddddddd

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import numpy as np
# import pandas as pd
# import os
# import json
# import matplotlib
# matplotlib.use('Agg')
# import matplotlib.pyplot as plt
# from sklearn.metrics import (
#     roc_auc_score, f1_score, accuracy_score,
#     classification_report, confusion_matrix
# )
# from sklearn.preprocessing import label_binarize
# from collections import defaultdict
# from torchvision import models, transforms
# from torch.utils.data import DataLoader, Dataset
# from PIL import Image
# from matplotlib.patches import Patch

# # ─────────────────────────────────────────────
# # 0. Device
# # ─────────────────────────────────────────────
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using: {DEVICE}")

# NUM_CLASSES = 5   # BI-RADS 0-4

# # ─────────────────────────────────────────────
# # 1. Label maps
# # ─────────────────────────────────────────────
# BIRADS_MAP = {
#     0: 'BI-RADS 1',
#     1: 'BI-RADS 2',
#     2: 'BI-RADS 3',
#     3: 'BI-RADS 4',
#     4: 'BI-RADS 5/6',
# }
# FINDING_MAP = {
#     0: 'No Finding',
#     1: 'Calcification',
#     2: 'Mass',
#     3: 'Mass+Calc/Lymph',
#     4: 'Structural/Complex',
# }
# DENSITY_MAP = {
#     0: 'Density A (Fatty)',
#     1: 'Density B (Scattered)',
#     2: 'Density C (Heterogeneous)',
#     3: 'Density D (Dense)',
# }
# DEVICE_MAP = {
#     0: 'Mammomat Inspiration',
#     1: 'Planmed Nuance',
#     2: 'GIOTTO',
# }

# PALETTE = {
#     ('efficientnet_b3', '512') : '#2196F3',
#     ('efficientnet_b3', '1024'): '#0D47A1',
#     ('convnext_base',   '512') : '#F44336',
#     ('convnext_base',   '1024'): '#B71C1C',
# }
# DISPLAY_NAMES = {
#     ('efficientnet_b3', '512') : 'EfficientNet-B3 (512)',
#     ('efficientnet_b3', '1024'): 'EfficientNet-B3 (1024)',
#     ('convnext_base',   '512') : 'ConvNeXt-Base (512)',
#     ('convnext_base',   '1024'): 'ConvNeXt-Base (1024)',
# }
# SPLIT_LABELS = {
#     'val'      : 'Validation',
#     'vindr'    : 'VinDr-Mammo Test',
#     'inbreast' : 'INbreast Test',
# }

# # ─────────────────────────────────────────────
# # 2. Model configs  — 5-class
# # ─────────────────────────────────────────────
# models_config = [
#     {
#         'name'      : 'efficientnet_b3',
#         'class'     : models.efficientnet_b3,
#         'weights'   : models.EfficientNet_B3_Weights.DEFAULT,
#         'ckpt_512'  : 'Thesis_updated_results/Merged_birads_512/efficientnet_b3/best_model.pth',
#         'ckpt_1024' : 'Thesis_updated_results/Merged_birads_1024/efficientnet_b3/best_model.pth',
#     },
#     {
#         'name'      : 'convnext_base',
#         'class'     : models.convnext_base,
#         'weights'   : models.ConvNeXt_Base_Weights.DEFAULT,
#         'ckpt_512'  : 'Thesis_updated_results/Merged_birads_512/convnext_base/best_model.pth',
#         'ckpt_1024' : 'Thesis_updated_results/Merged_birads_1024/convnext_base/best_model.pth',
#     },
# ]

# # ─────────────────────────────────────────────
# # 3. GenericNet  — num_classes=5
# # ─────────────────────────────────────────────
# class GenericNet(nn.Module):
#     def __init__(self, backbone_name, backbone_class, backbone_weights,
#                  num_classes=NUM_CLASSES):
#         super().__init__()
#         self.num_classes   = num_classes
#         self.backbone_name = backbone_name
#         self.backbone      = backbone_class(weights=backbone_weights)

#         if 'efficientnet' in backbone_name:
#             num_features = self.backbone.classifier[1].in_features
#             self.backbone.classifier = nn.Identity()
#         elif 'resnet' in backbone_name:
#             num_features = self.backbone.fc.in_features
#             self.backbone.fc = nn.Identity()
#         elif 'densenet' in backbone_name:
#             num_features = self.backbone.classifier.in_features
#             self.backbone.classifier = nn.Identity()
#         elif 'convnext' in backbone_name:
#             num_features = self.backbone.classifier[2].in_features
#             self.backbone.classifier = nn.Identity()
#         elif 'swin' in backbone_name:
#             num_features = self.backbone.head.in_features
#             self.backbone.head = nn.Identity()
#         else:
#             raise ValueError(f"Unsupported: {backbone_name}")

#         self.global_pool = nn.AdaptiveAvgPool2d(1)
#         hidden_size      = 768 if 'swin' in backbone_name else 512
#         self.classifier  = nn.Sequential(
#             nn.Linear(num_features, hidden_size), nn.ReLU(), nn.Dropout(0.4),
#             nn.Linear(hidden_size, 256),          nn.ReLU(), nn.Dropout(0.2),
#             nn.Linear(256, num_classes)
#         )

#     def forward(self, x):
#         features = self.backbone(x)
#         if isinstance(features, tuple):
#             features = features[0]
#         if features.ndim == 4:
#             features = self.global_pool(features).flatten(1)
#         return self.classifier(features)


# # ─────────────────────────────────────────────
# # 4. Dataset & DataLoaders
# # ─────────────────────────────────────────────
# class VinDrMammoDataset(Dataset):
#     def __init__(self, dataframe, val_transform, mode="val"):
#         self.df            = dataframe.reset_index(drop=True)
#         self.val_transform = val_transform
#         self.mode          = mode
#         counts = self.df["birads"].value_counts().sort_index().to_dict()
#         print(f"{mode.upper()} BIRADS distribution:", counts)

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row      = self.df.iloc[idx]
#         image    = Image.open(row["merged_image_path"]).convert("RGB")
#         birads   = int(row["birads"])
#         density  = int(row['density'])   if 'density'   in row else 0
#         finding  = int(row['finding'])   if 'finding'   in row else 0
#         device_id= int(row['device_id']) if 'device_id' in row else 0
#         return (
#             row["merged_image_path"],
#             self.val_transform(image),
#             torch.tensor(birads,    dtype=torch.long),
#             torch.tensor(density,   dtype=torch.long),
#             torch.tensor(finding,   dtype=torch.long),
#             torch.tensor(device_id, dtype=torch.long),
#         )


# def get_val_transform(img_size=(512, 512)):
#     return transforms.Compose([
#         transforms.Resize(img_size),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                              std=[0.229, 0.224, 0.225]),
#     ])


# def create_test_loaders(val_df, test_df, inbreast_df, batch_size=8):
#     t512  = get_val_transform((512,  512))
#     t1024 = get_val_transform((1024, 1024))
#     val_kw    = dict(batch_size=batch_size, shuffle=False,
#                      num_workers=12, pin_memory=True)
#     single_kw = dict(batch_size=1, shuffle=False,
#                      num_workers=12, pin_memory=True)
#     val_512       = DataLoader(VinDrMammoDataset(val_df,      t512,  'val'), **val_kw)
#     val_1024      = DataLoader(VinDrMammoDataset(val_df,      t1024, 'val'), **val_kw)
#     test_512      = DataLoader(VinDrMammoDataset(test_df,     t512,  'val'), **single_kw)
#     test_1024     = DataLoader(VinDrMammoDataset(test_df,     t1024, 'val'), **single_kw)
#     inbreast_512  = DataLoader(VinDrMammoDataset(inbreast_df, t512,  'val'), **single_kw)
#     inbreast_1024 = DataLoader(VinDrMammoDataset(inbreast_df, t1024, 'val'), **single_kw)
#     return val_512, val_1024, test_512, test_1024, inbreast_512, inbreast_1024


# val_512, val_1024, test_512, test_1024, inbreast_512, inbreast_1024 = create_test_loaders(
#     val_df, test_df, inbreast_df, batch_size=8
# )

# # ─────────────────────────────────────────────
# # 5. Utility helpers
# # ─────────────────────────────────────────────
# def to_py(v):
#     if isinstance(v, torch.Tensor): return v.item()
#     if isinstance(v, np.generic):   return v.item()
#     return v


# def safe_json(obj):
#     if isinstance(obj, dict):
#         return {str(k): safe_json(v) for k, v in obj.items()}
#     if isinstance(obj, (list, tuple)):
#         return [safe_json(i) for i in obj]
#     if isinstance(obj, torch.Tensor):
#         return obj.item() if obj.numel() == 1 else obj.tolist()
#     if isinstance(obj, np.ndarray):
#         return obj.tolist()
#     if isinstance(obj, (np.floating, np.integer)):
#         return obj.item()
#     if isinstance(obj, float) and np.isnan(obj):
#         return None
#     return obj


# # ─────────────────────────────────────────────
# # 6. Inference  — birads IS the label
# # ─────────────────────────────────────────────
# @torch.no_grad()
# def run_inference(model, loader, infer_device):
#     model.eval()
#     paths_l, labels_l, preds_l = [], [], []
#     probs_l, density_l, finding_l, devid_l = [], [], [], []

#     for batch in loader:
#         paths, images, birads, densities, findings, device_ids = batch
#         images = images.to(infer_device)
#         logits = model(images)
#         probs  = F.softmax(logits, dim=1)        # [B, 5]
#         preds  = logits.argmax(dim=1)

#         paths_l.extend(list(paths))
#         labels_l.extend([to_py(v) for v in birads])
#         preds_l.extend([to_py(v) for v in preds])
#         probs_l.extend([p.cpu().numpy() for p in probs])
#         density_l.extend([to_py(v) for v in densities])
#         finding_l.extend([to_py(v) for v in findings])
#         devid_l.extend([to_py(v) for v in device_ids])

#     probs_arr = np.stack(probs_l, axis=0)        # [N, 5]
#     df = pd.DataFrame({
#         'path'      : paths_l,
#         'label'     : labels_l,   # true BIRADS 0-4
#         'pred'      : preds_l,    # predicted BIRADS 0-4
#         'density'   : density_l,
#         'finding'   : finding_l,
#         'device_id' : devid_l,
#     })
#     for c in range(NUM_CLASSES):
#         df[f'prob_cls{c}'] = probs_arr[:, c]
#     return df


# # ─────────────────────────────────────────────
# # 7. Metrics
# # ─────────────────────────────────────────────
# def multiclass_metrics(df):
#     lbl       = df['label'].values
#     prd       = df['pred'].values
#     prob_cols = [f'prob_cls{c}' for c in range(NUM_CLASSES)]
#     probs     = df[prob_cols].values
#     present   = np.unique(lbl)

#     try:
#         if len(present) == NUM_CLASSES:
#             auc = roc_auc_score(lbl, probs, multi_class='ovr', average='macro')
#         else:
#             probs_sub = probs[:, present]
#             probs_sub = probs_sub / probs_sub.sum(axis=1, keepdims=True)
#             auc = roc_auc_score(label_binarize(lbl, classes=present),
#                                 probs_sub, multi_class='ovr', average='macro')
#     except Exception:
#         auc = float('nan')

#     per_class_f1 = {
#         BIRADS_MAP[c]: round(float(
#             f1_score(lbl == c, prd == c, average='binary', zero_division=0)
#         ), 4)
#         for c in range(NUM_CLASSES)
#     }
#     return {
#         'n_total'     : int(len(df)),
#         'accuracy'    : round(float(accuracy_score(lbl, prd)), 4),
#         'f1_macro'    : round(float(f1_score(lbl, prd, average='macro',
#                                              zero_division=0)), 4),
#         'f1_weighted' : round(float(f1_score(lbl, prd, average='weighted',
#                                              zero_division=0)), 4),
#         'auc_macro'   : round(float(auc), 4) if not np.isnan(auc) else None,
#         'per_class_f1': per_class_f1,
#     }


# def subgroup_metrics_mc(df, group_col, label_map=None):
#     rows = []
#     for val in sorted(df[group_col].unique()):
#         grp  = df[df[group_col] == val]
#         lbl  = grp['label'].values
#         prd  = grp['pred'].values
#         name = label_map.get(int(val), str(val)) if label_map else str(val)
#         rows.append({
#             'group'      : name,
#             'n_total'    : len(grp),
#             'accuracy'   : round(float(accuracy_score(lbl, prd)), 4),
#             'f1_macro'   : round(float(f1_score(lbl, prd, average='macro',
#                                                 zero_division=0)), 4),
#             'f1_weighted': round(float(f1_score(lbl, prd, average='weighted',
#                                                 zero_division=0)), 4),
#         })
#     return pd.DataFrame(rows)


# # ─────────────────────────────────────────────
# # 8. Plot helpers
# # ─────────────────────────────────────────────

# # ── 8a. 5×5 Confusion matrix ─────────────────
# def plot_confusion_matrix_mc(cm, class_names, title, save_path, dpi=300):
#     n       = len(class_names)
#     cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
#     fig, ax = plt.subplots(figsize=(n * 1.6 + 1.5, n * 1.4 + 1.5))
#     im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
#     plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03,
#                  label='Row-normalised proportion')
#     ax.set_xticks(range(n))
#     ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=10)
#     ax.set_yticks(range(n))
#     ax.set_yticklabels(class_names, fontsize=10)
#     ax.set_xlabel('Predicted BIRADS', fontsize=12)
#     ax.set_ylabel('True BIRADS', fontsize=12)
#     ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
#     for i in range(n):
#         for j in range(n):
#             ax.text(j, i,
#                     f'{cm[i,j]}\n({cm_norm[i,j]*100:.0f}%)',
#                     ha='center', va='center', fontsize=8, fontweight='bold',
#                     color='white' if cm_norm[i, j] > 0.5 else 'black')
#     plt.tight_layout()
#     plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8b. Group → predicted BIRADS matrix ──────
# def plot_group_vs_pred_birads(df, group_col, group_map,
#                                title, save_path, dpi=300):
#     """
#     Rows = group categories  (Finding / Density / Device)
#     Cols = Predicted BIRADS class (0-4)
#     Colour = YlOrRd by row-normalised proportion  (hot = high density)
#     Shows where each group's samples end up in predictions.
#     """
#     group_vals  = sorted(df[group_col].unique())
#     birads_vals = list(range(NUM_CLASSES))

#     mat = np.zeros((len(group_vals), NUM_CLASSES), dtype=int)
#     for i, gv in enumerate(group_vals):
#         for j, bv in enumerate(birads_vals):
#             mat[i, j] = int(((df[group_col] == gv) & (df['pred'] == bv)).sum())

#     mat_norm   = mat.astype(float) / mat.sum(axis=1, keepdims=True).clip(min=1)
#     row_labels = [group_map.get(int(v), str(v)) for v in group_vals]
#     col_labels = [BIRADS_MAP[c] for c in birads_vals]

#     fig, ax = plt.subplots(figsize=(NUM_CLASSES * 1.8 + 1,
#                                     len(group_vals) * 1.0 + 1.5))
#     im = ax.imshow(mat_norm, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
#     plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03, label='Row proportion')
#     ax.set_xticks(range(NUM_CLASSES))
#     ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=10)
#     ax.set_yticks(range(len(row_labels)))
#     ax.set_yticklabels(row_labels, fontsize=10)
#     ax.set_xlabel('Predicted BIRADS', fontsize=12)
#     ax.set_ylabel(group_col.replace('_', ' ').title(), fontsize=12)
#     ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
#     for i in range(len(group_vals)):
#         for j in range(NUM_CLASSES):
#             ax.text(j, i,
#                     f'{mat[i,j]}\n({mat_norm[i,j]*100:.0f}%)',
#                     ha='center', va='center', fontsize=8, fontweight='bold',
#                     color='white' if mat_norm[i, j] > 0.55 else 'black')
#     plt.tight_layout()
#     plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8c. Group → TRUE BIRADS matrix ───────────
# def plot_group_vs_true_birads(df, group_col, group_map,
#                                title, save_path, dpi=300):
#     """Same structure but cols = True BIRADS (GT distribution)."""
#     group_vals  = sorted(df[group_col].unique())
#     birads_vals = list(range(NUM_CLASSES))

#     mat = np.zeros((len(group_vals), NUM_CLASSES), dtype=int)
#     for i, gv in enumerate(group_vals):
#         for j, bv in enumerate(birads_vals):
#             mat[i, j] = int(((df[group_col] == gv) & (df['label'] == bv)).sum())

#     mat_norm   = mat.astype(float) / mat.sum(axis=1, keepdims=True).clip(min=1)
#     row_labels = [group_map.get(int(v), str(v)) for v in group_vals]
#     col_labels = [BIRADS_MAP[c] for c in birads_vals]

#     fig, ax = plt.subplots(figsize=(NUM_CLASSES * 1.8 + 1,
#                                     len(group_vals) * 1.0 + 1.5))
#     im = ax.imshow(mat_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
#     plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03, label='Row proportion')
#     ax.set_xticks(range(NUM_CLASSES))
#     ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=10)
#     ax.set_yticks(range(len(row_labels)))
#     ax.set_yticklabels(row_labels, fontsize=10)
#     ax.set_xlabel('True BIRADS', fontsize=12)
#     ax.set_ylabel(group_col.replace('_', ' ').title(), fontsize=12)
#     ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
#     for i in range(len(group_vals)):
#         for j in range(NUM_CLASSES):
#             ax.text(j, i,
#                     f'{mat[i,j]}\n({mat_norm[i,j]*100:.0f}%)',
#                     ha='center', va='center', fontsize=8, fontweight='bold',
#                     color='white' if mat_norm[i, j] > 0.55 else 'black')
#     plt.tight_layout()
#     plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8d. Hierarchical error analysis (multiclass) ──
# def plot_hierarchical_error_analysis_mc(df, group_col, group_map,
#                                          model_name, split_name,
#                                          resolution, out_dir, dpi=300):
#     """
#     For each group value, compute:
#       - correct: pred == label
#       - exact-1 off: |pred - label| == 1  (adjacent misclassification)
#       - severe:  |pred - label| >= 2
#     Produces a 4-panel figure + CSV.
#     """
#     results = []
#     for val in sorted(df[group_col].unique()):
#         grp   = df[df[group_col] == val]
#         total = len(grp)
#         lbl   = grp['label'].values
#         prd   = grp['pred'].values
#         diff  = np.abs(prd.astype(int) - lbl.astype(int))

#         correct    = int((diff == 0).sum())
#         adj_err    = int((diff == 1).sum())
#         severe_err = int((diff >= 2).sum())

#         # direction breakdown
#         overpredict = int((prd > lbl).sum())   # predicted higher BIRADS
#         underpredict= int((prd < lbl).sum())   # predicted lower BIRADS

#         results.append({
#             'group_val'    : val,
#             'group_name'   : group_map.get(int(val), str(val)),
#             'total'        : total,
#             'correct'      : correct,
#             'adj_err'      : adj_err,
#             'severe_err'   : severe_err,
#             'accuracy'     : round(correct / total, 4) if total > 0 else 0,
#             'adj_err_rate' : round(adj_err  / total, 4) if total > 0 else 0,
#             'sev_err_rate' : round(severe_err/total, 4) if total > 0 else 0,
#             'overpredict'  : overpredict,
#             'underpredict' : underpredict,
#         })

#     res_df = pd.DataFrame(results)
#     n      = len(res_df)
#     names  = res_df['group_name'].tolist()
#     x      = np.arange(n)

#     fig, axes = plt.subplots(2, 2, figsize=(15, 10))

#     # Panel 1: Accuracy per group
#     ax = axes[0, 0]
#     colors_acc = ['#4CAF50' if v >= 0.7 else '#FFC107' if v >= 0.5 else '#F44336'
#                   for v in res_df['accuracy']]
#     bars = ax.barh(names, res_df['accuracy'], color=colors_acc)
#     ax.set_xlim(0, 1.12)
#     ax.set_xlabel('Accuracy', fontsize=11)
#     ax.set_title('Accuracy by Group', fontsize=12, fontweight='bold')
#     ax.grid(axis='x', alpha=0.3)
#     for bar, acc, tot in zip(bars, res_df['accuracy'], res_df['total']):
#         ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2,
#                 f'{acc:.2%}  (n={tot})', va='center', fontsize=8)

#     # Panel 2: Error type stacked bar
#     ax = axes[0, 1]
#     w = 0.55
#     ax.bar(x, res_df['correct'],    width=w, label='Correct',       color='#4CAF50')
#     ax.bar(x, res_df['adj_err'],    width=w, label='Adjacent (±1)', color='#FFA726',
#            bottom=res_df['correct'])
#     ax.bar(x, res_df['severe_err'], width=w, label='Severe (≥2)',   color='#F44336',
#            bottom=res_df['correct'] + res_df['adj_err'])
#     ax.set_ylabel('Count', fontsize=11)
#     ax.set_title('Error Type Breakdown', fontsize=12, fontweight='bold')
#     ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
#     ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

#     # Panel 3: Over/Under prediction
#     ax = axes[1, 0]
#     ax.bar(x - 0.2, res_df['overpredict'],  width=0.35,
#            label='Overpredicted (↑ BIRADS)', color='#E91E63', alpha=0.85)
#     ax.bar(x + 0.2, res_df['underpredict'], width=0.35,
#            label='Underpredicted (↓ BIRADS)', color='#2196F3', alpha=0.85)
#     ax.set_ylabel('Count', fontsize=11)
#     ax.set_title('Over vs Under Prediction', fontsize=12, fontweight='bold')
#     ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
#     ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

#     # Panel 4: Summary table
#     ax = axes[1, 1]
#     ax.axis('off')
#     table_data = []
#     for _, row in res_df.iterrows():
#         status = 'GOOD' if row['accuracy'] >= 0.7 else 'WARN' if row['accuracy'] >= 0.5 else 'POOR'
#         table_data.append([
#             row['group_name'],
#             str(row['total']),
#             f"{row['accuracy']:.1%}",
#             str(row['adj_err']),
#             str(row['severe_err']),
#             str(row['overpredict']),
#             str(row['underpredict']),
#             status
#         ])
#     table = ax.table(
#         cellText=table_data,
#         colLabels=['Group', 'N', 'Acc', 'Adj±1', 'Sev≥2', 'Over↑', 'Under↓', 'Status'],
#         cellLoc='center', loc='center',
#         colWidths=[0.28, 0.08, 0.1, 0.09, 0.09, 0.09, 0.09, 0.1]
#     )
#     table.auto_set_font_size(False)
#     table.set_fontsize(8)
#     table.scale(1, 2.2)
#     for i, row in res_df.iterrows():
#         color = '#C8E6C9' if row['accuracy'] >= 0.7 else '#FFF9C4' if row['accuracy'] >= 0.5 else '#FFCDD2'
#         table[(i + 1, 7)].set_facecolor(color)
#     ax.set_title('Summary Table', fontsize=12, fontweight='bold', pad=20)

#     safe_col = group_col.replace('_', ' ').title()
#     fig.suptitle(
#         f'BIRADS Prediction Error Analysis by {safe_col}\n'
#         f'{DISPLAY_NAMES.get((model_name, resolution), model_name)} | '
#         f'{SPLIT_LABELS.get(split_name, split_name)}',
#         fontsize=14, fontweight='bold'
#     )
#     plt.tight_layout()
#     fname = os.path.join(
#         out_dir,
#         f'{model_name}_{split_name}_{resolution}_{group_col}_error_analysis.png'
#     )
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()

#     csv_fname = fname.replace('.png', '.csv')
#     res_df.to_csv(csv_fname, index=False)
#     print(f"\n  Error analysis ({safe_col}):")
#     print(res_df[['group_name','total','accuracy','adj_err','severe_err',
#                   'overpredict','underpredict']].to_string(index=False))
#     return res_df


# # ── 8e. Per-class F1 bar ─────────────────────
# def plot_per_class_f1_bar(per_class_f1, model_name, split_name,
#                            resolution, out_dir, dpi=300):
#     classes = list(per_class_f1.keys())
#     vals    = list(per_class_f1.values())
#     colors  = ['#42A5F5','#66BB6A','#FFA726','#EF5350','#AB47BC']
#     fig, ax = plt.subplots(figsize=(9, 5))
#     bars = ax.bar(classes, vals, color=colors[:len(classes)], width=0.55)
#     ax.set_ylim(0, 1.12)
#     ax.set_ylabel('F1 Score (OvR)', fontsize=12)
#     ax.set_title(
#         f'Per-BIRADS F1 — '
#         f'{DISPLAY_NAMES.get((model_name,resolution),model_name)}\n'
#         f'{SPLIT_LABELS.get(split_name,split_name)}',
#         fontsize=13, fontweight='bold'
#     )
#     ax.grid(axis='y', alpha=0.3, linestyle='--')
#     for bar, val in zip(bars, vals):
#         ax.text(bar.get_x() + bar.get_width()/2,
#                 bar.get_height() + 0.02,
#                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
#     plt.xticks(rotation=20, ha='right', fontsize=10)
#     plt.tight_layout()
#     fname = os.path.join(out_dir,
#                          f'{model_name}_{split_name}_{resolution}_per_class_f1.png')
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8f. 2-panel subgroup bar ─────────────────
# def plot_subgroup_panel_mc(sg_df, group_name, model_name, split_name,
#                             resolution, out_dir, dpi=300):
#     n   = len(sg_df)
#     fig, axes = plt.subplots(1, 2, figsize=(14, max(4, n * 0.65 + 1.5)))
#     for ax, metric, color, title in zip(
#         axes,
#         ['accuracy', 'f1_macro'],
#         ['#1976D2', '#F57C00'],
#         ['Accuracy', 'F1 Macro']
#     ):
#         vals  = sg_df[metric].fillna(0).values
#         names = sg_df['group'].values
#         y     = np.arange(len(names))
#         bars  = ax.barh(y, vals, color=color, height=0.55)
#         ax.set_yticks(y); ax.set_yticklabels(names, fontsize=10)
#         ax.set_xlim(0, 1.15)
#         ax.set_xlabel(title, fontsize=11)
#         ax.set_title(title, fontsize=12, fontweight='bold')
#         ax.grid(axis='x', alpha=0.3, linestyle='--')
#         for bar, val in zip(bars, vals):
#             ax.text(bar.get_width() + 0.01,
#                     bar.get_y() + bar.get_height()/2,
#                     f'{val:.3f}', va='center', fontsize=9)
#         for yi, row in enumerate(sg_df.itertuples()):
#             ax.text(-0.02, yi, f'n={row.n_total}',
#                     va='center', ha='right', fontsize=8, color='grey')
#     fig.suptitle(
#         f'{DISPLAY_NAMES.get((model_name,resolution),model_name)} | '
#         f'{SPLIT_LABELS.get(split_name,split_name)} | {group_name}',
#         fontsize=13, fontweight='bold', y=1.01
#     )
#     plt.tight_layout()
#     safe = group_name.lower().replace(' ', '_').replace('-', '')
#     fname = os.path.join(out_dir,
#                          f'{model_name}_{split_name}_{resolution}_{safe}_panel.png')
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8g. Subgroup macro-AUC bar ───────────────
# def plot_subgroup_auc_mc(df, group_col, group_map,
#                           model_name, split_name, resolution,
#                           out_dir, dpi=300):
#     group_vals = sorted(df[group_col].unique())
#     prob_cols  = [f'prob_cls{c}' for c in range(NUM_CLASSES)]
#     names, aucs = [], []

#     for gv in group_vals:
#         grp  = df[df[group_col] == gv]
#         lbl  = grp['label'].values
#         prbs = grp[prob_cols].values
#         name = group_map.get(int(gv), str(gv))
#         try:
#             present = np.unique(lbl)
#             if len(present) < 2:
#                 continue
#             if len(present) == NUM_CLASSES:
#                 auc = roc_auc_score(lbl, prbs, multi_class='ovr', average='macro')
#             else:
#                 prbs_sub = prbs[:, present]
#                 prbs_sub = prbs_sub / prbs_sub.sum(axis=1, keepdims=True)
#                 auc = roc_auc_score(label_binarize(lbl, classes=present),
#                                     prbs_sub, multi_class='ovr', average='macro')
#             names.append(name); aucs.append(round(float(auc), 4))
#         except Exception:
#             pass

#     if not names:
#         return

#     colors = plt.cm.tab10(np.linspace(0, 0.9, len(names)))
#     fig, ax = plt.subplots(figsize=(max(8, len(names) * 1.3 + 1), 5))
#     bars = ax.bar(names, aucs, color=colors, width=0.55)
#     ax.set_ylim(0, 1.12)
#     ax.set_ylabel('Macro AUC (OvR)', fontsize=12)
#     safe_col = group_col.replace('_', ' ').title()
#     ax.set_title(
#         f'AUC by {safe_col} — '
#         f'{DISPLAY_NAMES.get((model_name,resolution),model_name)}\n'
#         f'{SPLIT_LABELS.get(split_name,split_name)}',
#         fontsize=13, fontweight='bold'
#     )
#     ax.grid(axis='y', alpha=0.3, linestyle='--')
#     for bar, val in zip(bars, aucs):
#         ax.text(bar.get_x() + bar.get_width()/2,
#                 bar.get_height() + 0.01,
#                 f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
#     plt.xticks(rotation=20, ha='right', fontsize=10)
#     plt.tight_layout()
#     fname = os.path.join(
#         out_dir,
#         f'{model_name}_{split_name}_{resolution}_auc_by_{group_col}.png'
#     )
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()


# # ── 8h. Single model OvR ROC (5 curves) ──────
# def plot_single_auc_mc(df, model_name, split_name, resolution,
#                         out_dir, dpi=300):
#     from sklearn.metrics import roc_curve as _rc
#     lbl       = df['label'].values
#     prob_cols = [f'prob_cls{c}' for c in range(NUM_CLASSES)]
#     probs     = df[prob_cols].values
#     present   = sorted(df['label'].unique())
#     colors    = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']

#     fig, ax = plt.subplots(figsize=(8, 7))
#     for c in range(NUM_CLASSES):
#         if c not in present:
#             continue
#         binary = (lbl == c).astype(int)
#         if binary.sum() == 0 or binary.sum() == len(binary):
#             continue
#         fpr, tpr, _ = _rc(binary, probs[:, c])
#         auc_val      = roc_auc_score(binary, probs[:, c])
#         ax.plot(fpr, tpr, lw=2.2, color=colors[c],
#                 label=f'{BIRADS_MAP[c]}  AUC={auc_val:.3f}')

#     ax.plot([0,1],[0,1],'k--', lw=1.1, label='Random')
#     ax.set_xlabel('False Positive Rate', fontsize=13)
#     ax.set_ylabel('True Positive Rate', fontsize=13)
#     ax.set_title(
#         f'OvR ROC — {DISPLAY_NAMES.get((model_name,resolution),model_name)}\n'
#         f'{SPLIT_LABELS.get(split_name,split_name)}',
#         fontsize=14, fontweight='bold'
#     )
#     ax.legend(fontsize=10, loc='lower right')
#     ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
#     ax.grid(alpha=0.25, linestyle='--')
#     for sp in ax.spines.values():
#         sp.set_linewidth(1.2)
#     plt.tight_layout()
#     fname = os.path.join(out_dir,
#                          f'{model_name}_{split_name}_{resolution}_roc.png')
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()

#     try:
#         auc_macro = roc_auc_score(
#             label_binarize(lbl, classes=list(range(NUM_CLASSES))),
#             probs, multi_class='ovr', average='macro'
#         )
#     except Exception:
#         auc_macro = float('nan')
#     return auc_macro


# # ── 8i. Combined macro-AUC bar ───────────────
# def plot_combined_auc_mc(entries, split_name, out_dir, dpi=300):
#     fig, ax = plt.subplots(figsize=(9, 5))
#     names  = [DISPLAY_NAMES.get((e['model'], e['resolution']), e['model'])
#                for e in entries]
#     aucs   = [e['auc_macro'] for e in entries]
#     colors = [PALETTE.get((e['model'], e['resolution']), '#555') for e in entries]
#     bars   = ax.bar(names, aucs, color=colors, width=0.5)
#     ax.set_ylim(0, 1.12)
#     ax.set_ylabel('Macro AUC (OvR)', fontsize=12)
#     ax.set_title(
#         f'Macro AUC Comparison — {SPLIT_LABELS.get(split_name,split_name)}',
#         fontsize=14, fontweight='bold'
#     )
#     ax.grid(axis='y', alpha=0.3, linestyle='--')
#     for bar, val in zip(bars, aucs):
#         ax.text(bar.get_x() + bar.get_width()/2,
#                 bar.get_height() + 0.01,
#                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
#     plt.xticks(rotation=15, ha='right', fontsize=10)
#     plt.tight_layout()
#     fname = os.path.join(out_dir, f'combined_auc_{split_name}.png')
#     plt.savefig(fname, dpi=dpi, bbox_inches='tight')
#     plt.close()
#     print(f"  Combined AUC → {fname}")


# # ─────────────────────────────────────────────
# # 9. Text report
# # ─────────────────────────────────────────────
# def write_text_report(summary, cls_rep, cm, class_names,
#                        density_sg, device_sg, finding_sg,
#                        density_err, device_err, finding_err,
#                        is_inbreast, out_dir, px):
#     lines = []
#     sep   = '=' * 65

#     lines += [sep,
#               'BIRADS 5-CLASS CLASSIFICATION — RESULTS REPORT',
#               f"Model      : {summary['model']}",
#               f"Split      : {SPLIT_LABELS.get(summary['split'], summary['split'])}",
#               f"Resolution : {summary['resolution']}",
#               sep, '']

#     ov = summary['overall']
#     lines += ["── OVERALL METRICS ──────────────────────────────────",
#               f"  N Total     : {ov['n_total']}",
#               f"  Accuracy    : {ov['accuracy']}",
#               f"  F1 Macro    : {ov['f1_macro']}",
#               f"  F1 Weighted : {ov['f1_weighted']}",
#               f"  AUC Macro   : {ov['auc_macro']}",
#               '', '  Per-Class F1:']
#     for cls, f1v in ov['per_class_f1'].items():
#         lines.append(f"    {cls:15s}: {f1v:.4f}")

#     lines += ['',
#               "── CLASSIFICATION REPORT ───────────────────────────",
#               cls_rep,
#               "── CONFUSION MATRIX (rows=True, cols=Pred) ─────────"]
#     header = '              ' + '  '.join(f'{n:>12}' for n in class_names)
#     lines.append(header)
#     for i, rn in enumerate(class_names):
#         row_str = f'{rn:14s}' + '  '.join(f'{cm[i,j]:12d}'
#                                             for j in range(len(class_names)))
#         lines.append(row_str)
#     lines.append('')

#     def _err_block(title, sg_df, err_df):
#         block = [f"── {title} ──────────────────────────────────────────"]
#         if sg_df is not None and len(sg_df):
#             block.append(sg_df.to_string(index=False))
#         if err_df is not None and len(err_df):
#             block += ['', '  Error Analysis (adj=|pred-label|=1, sev=|pred-label|>=2):']
#             for _, row in err_df.iterrows():
#                 block.append(
#                     f"    {row['group_name']:25s}: Acc={row['accuracy']:.2%}  "
#                     f"Adj={row['adj_err']}  Sev={row['severe_err']}  "
#                     f"Over={row['overpredict']}  Under={row['underpredict']}"
#                 )
#         block.append('')
#         return block

#     if not is_inbreast:
#         lines += _err_block('PER-DENSITY', density_sg, density_err)
#         lines += _err_block('PER-DEVICE',  device_sg,  device_err)
#         lines += _err_block('PER-FINDING', finding_sg, finding_err)

#     lines.append(sep)
#     fpath = os.path.join(out_dir, f'{px}_report.txt')
#     with open(fpath, 'w') as f:
#         f.write('\n'.join(lines))
#     print(f"  Text report → {fpath}")


# # ─────────────────────────────────────────────
# # 10. Main evaluation
# # ─────────────────────────────────────────────
# _auc_collector = defaultdict(list)


# def evaluate_and_save(model, df_res, model_name, split_name, resolution, out_dir):
#     os.makedirs(out_dir, exist_ok=True)
#     px          = f"{model_name}_{split_name}_{resolution}"
#     is_inbreast = (split_name == 'inbreast')
#     dn          = DISPLAY_NAMES.get((model_name, resolution), model_name)
#     sl          = SPLIT_LABELS.get(split_name, split_name)
#     class_names = [BIRADS_MAP[c] for c in range(NUM_CLASSES)]

#     # ── Overall ──────────────────────────────
#     overall = multiclass_metrics(df_res)
#     cls_rep = classification_report(
#         df_res['label'], df_res['pred'],
#         target_names=class_names, zero_division=0
#     )
#     cm = confusion_matrix(df_res['label'], df_res['pred'],
#                           labels=list(range(NUM_CLASSES)))

#     print(f"\n{'─'*65}")
#     print(f"  {dn} | {sl}")
#     print(f"  Acc={overall['accuracy']} | F1-Macro={overall['f1_macro']} "
#           f"| AUC={overall['auc_macro']}")
#     print(cls_rep)

#     # 5×5 confusion matrix
#     plot_confusion_matrix_mc(
#         cm, class_names,
#         f'Confusion Matrix — {dn}\n{sl}',
#         os.path.join(out_dir, f'{px}_confusion_matrix.png')
#     )

#     # Per-class F1
#     plot_per_class_f1_bar(
#         overall['per_class_f1'], model_name, split_name, resolution, out_dir
#     )

#     # Individual OvR ROC
#     auc_macro = plot_single_auc_mc(
#         df_res, model_name, split_name, resolution, out_dir
#     )
#     if auc_macro is not None and not np.isnan(auc_macro):
#         _auc_collector[split_name].append(dict(
#             model=model_name, resolution=resolution, auc_macro=auc_macro
#         ))

#     # ── VinDr / Val: full subgroup suite ─────
#     density_sg = device_sg = finding_sg = None
#     density_err= device_err= finding_err= None
#     density_rec= device_rec= finding_rec= []
#     density_err_rec= device_err_rec= finding_err_rec= []

#     if not is_inbreast:

#         for group_col, group_map, tag in [
#             ('finding',   FINDING_MAP, 'finding'),
#             ('density',   DENSITY_MAP, 'density'),
#             ('device_id', DEVICE_MAP,  'device'),
#         ]:
#             # predicted BIRADS matrix
#             plot_group_vs_pred_birads(
#                 df_res, group_col, group_map,
#                 f'{group_col.replace("_"," ").title()} → Predicted BIRADS — {dn}\n{sl}',
#                 os.path.join(out_dir, f'{px}_{tag}_vs_pred_birads.png')
#             )
#             # true BIRADS matrix
#             plot_group_vs_true_birads(
#                 df_res, group_col, group_map,
#                 f'{group_col.replace("_"," ").title()} → True BIRADS — {dn}\n{sl}',
#                 os.path.join(out_dir, f'{px}_{tag}_vs_true_birads.png')
#             )

#         # Per-Finding subgroup + AUC + error analysis
#         finding_sg = subgroup_metrics_mc(df_res, 'finding', FINDING_MAP)
#         finding_sg.to_csv(os.path.join(out_dir, f'{px}_finding.csv'), index=False)
#         plot_subgroup_panel_mc(finding_sg, 'Per-Finding',
#                                model_name, split_name, resolution, out_dir)
#         plot_subgroup_auc_mc(df_res, 'finding', FINDING_MAP,
#                               model_name, split_name, resolution, out_dir)
#         finding_err = plot_hierarchical_error_analysis_mc(
#             df_res, 'finding', FINDING_MAP,
#             model_name, split_name, resolution, out_dir
#         )
#         print("\n  Per-Finding:"); print(finding_sg.to_string(index=False))

#         # Per-Density subgroup + AUC + error analysis
#         density_sg = subgroup_metrics_mc(df_res, 'density', DENSITY_MAP)
#         density_sg.to_csv(os.path.join(out_dir, f'{px}_density.csv'), index=False)
#         plot_subgroup_panel_mc(density_sg, 'Per-Density',
#                                model_name, split_name, resolution, out_dir)
#         plot_subgroup_auc_mc(df_res, 'density', DENSITY_MAP,
#                               model_name, split_name, resolution, out_dir)
#         density_err = plot_hierarchical_error_analysis_mc(
#             df_res, 'density', DENSITY_MAP,
#             model_name, split_name, resolution, out_dir
#         )
#         print("\n  Per-Density:"); print(density_sg.to_string(index=False))

#         # Per-Device subgroup + AUC + error analysis
#         device_sg = subgroup_metrics_mc(df_res, 'device_id', DEVICE_MAP)
#         device_sg.to_csv(os.path.join(out_dir, f'{px}_device.csv'), index=False)
#         plot_subgroup_panel_mc(device_sg, 'Per-Device',
#                                model_name, split_name, resolution, out_dir)
#         plot_subgroup_auc_mc(df_res, 'device_id', DEVICE_MAP,
#                               model_name, split_name, resolution, out_dir)
#         device_err = plot_hierarchical_error_analysis_mc(
#             df_res, 'device_id', DEVICE_MAP,
#             model_name, split_name, resolution, out_dir
#         )
#         print("\n  Per-Device:"); print(device_sg.to_string(index=False))

#         density_rec     = density_sg.to_dict(orient='records')
#         device_rec      = device_sg.to_dict(orient='records')
#         finding_rec     = finding_sg.to_dict(orient='records')
#         density_err_rec = density_err.to_dict(orient='records') if density_err is not None else []
#         device_err_rec  = device_err.to_dict(orient='records')  if device_err  is not None else []
#         finding_err_rec = finding_err.to_dict(orient='records')  if finding_err is not None else []

#     else:
#         print("\n  [INbreast] Skipping density/device/finding.")

#     # ── Text report ──────────────────────────
#     write_text_report(
#         {'model': model_name, 'split': split_name,
#          'resolution': resolution, 'overall': overall},
#         cls_rep, cm, class_names,
#         density_sg, device_sg, finding_sg,
#         density_err, device_err, finding_err,
#         is_inbreast, out_dir, px
#     )

#     # ── JSON summary ─────────────────────────
#     summary = {
#         'model': model_name, 'split': split_name, 'resolution': resolution,
#         'overall'            : overall,
#         'per_density'        : density_rec,
#         'per_density_errors' : density_err_rec,
#         'per_device'         : device_rec,
#         'per_device_errors'  : device_err_rec,
#         'per_finding'        : finding_rec,
#         'per_finding_errors' : finding_err_rec,
#     }
#     with open(os.path.join(out_dir, f'{px}_summary.json'), 'w') as f:
#         json.dump(safe_json(summary), f, indent=2)

#     df_res.to_csv(os.path.join(out_dir, f'{px}_predictions.csv'), index=False)
#     print(f"  Saved → {out_dir}")
#     return summary


# # ─────────────────────────────────────────────
# # 11. Dataloaders map
# # ─────────────────────────────────────────────
# LOADERS = {
#     ('val',      '512')  : val_512,
#     ('val',      '1024') : val_1024,
#     ('vindr',    '512')  : test_512,
#     ('vindr',    '1024') : test_1024,
#     ('inbreast', '512')  : inbreast_512,
#     ('inbreast', '1024') : inbreast_1024,
# }

# COMBINED_DIR = 'Thesis_updated_results/BIRADS_results/combined_auc_plots'
# os.makedirs(COMBINED_DIR, exist_ok=True)

# # ─────────────────────────────────────────────
# # 12. Master loop
# # ─────────────────────────────────────────────
# all_summaries = []

# for cfg in models_config:
#     model_name = cfg['name']
#     print(f"\n{'='*70}\n  MODEL: {model_name}\n{'='*70}")

#     for resolution, ckpt_key in [('512','ckpt_512'), ('1024','ckpt_1024')]:
#         ckpt_path = cfg[ckpt_key]
#         if not os.path.exists(ckpt_path):
#             print(f"  [SKIP] Not found: {ckpt_path}")
#             continue

#         model = GenericNet(model_name, cfg['class'], cfg['weights']).to(DEVICE)
#         model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
#         model.eval()
#         print(f"\n  Loaded {resolution}: {ckpt_path}")

#         for (split_name, res), loader in LOADERS.items():
#             if res != resolution:
#                 continue
#             print(f"\n  → Inference: {split_name} | {resolution}")
#             df_res = run_inference(model, loader, DEVICE)
#             out_dir = (
#                 f"Thesis_updated_results/BIRADS_results/Merged_birads_{resolution}"
#                 f"/{model_name}/{split_name}"
#             )
#             summary = evaluate_and_save(
#                 model, df_res, model_name, split_name, resolution, out_dir
#             )
#             all_summaries.append(summary)

#         del model
#         torch.cuda.empty_cache()

# # ─────────────────────────────────────────────
# # 13. Combined AUC bar per split
# # ─────────────────────────────────────────────
# print(f"\n{'='*70}\nGenerating combined AUC plots ...")
# for split_name, entries in _auc_collector.items():
#     if len(entries) >= 2:
#         plot_combined_auc_mc(entries, split_name, COMBINED_DIR, dpi=300)

# # ─────────────────────────────────────────────
# # 14. Final side-by-side: test + inbreast
# # ─────────────────────────────────────────────
# print("\nGenerating final combined test + inbreast AUC ...")
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# for ax, target_split in zip(axes, ['vindr', 'inbreast']):
#     entries = _auc_collector.get(target_split, [])
#     names   = [DISPLAY_NAMES.get((e['model'], e['resolution']), e['model'])
#                for e in entries]
#     aucs    = [e['auc_macro'] for e in entries]
#     colors  = [PALETTE.get((e['model'], e['resolution']), '#555') for e in entries]
#     bars    = ax.bar(names, aucs, color=colors, width=0.5)
#     ax.set_ylim(0, 1.12)
#     ax.set_ylabel('Macro AUC (OvR)', fontsize=12)
#     ax.set_title(SPLIT_LABELS.get(target_split, target_split),
#                  fontsize=14, fontweight='bold')
#     ax.grid(axis='y', alpha=0.3, linestyle='--')
#     for bar, val in zip(bars, aucs):
#         ax.text(bar.get_x() + bar.get_width()/2,
#                 bar.get_height() + 0.01,
#                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
#     plt.sca(ax); plt.xticks(rotation=15, ha='right', fontsize=10)

# fig.suptitle('All Models — BIRADS Macro AUC (Test & INbreast)',
#              fontsize=15, fontweight='bold', y=1.01)
# plt.tight_layout()
# final_path = os.path.join(COMBINED_DIR, 'final_combined_test_inbreast_auc.png')
# plt.savefig(final_path, dpi=300, bbox_inches='tight')
# plt.close()
# print(f"  Final combined → {final_path}")

# # ─────────────────────────────────────────────
# # 15. Master comparison table
# # ─────────────────────────────────────────────
# rows = []
# for s in all_summaries:
#     row = {
#         'model'      : s['model'],
#         'split'      : s['split'],
#         'resolution' : s['resolution'],
#         'n_total'    : s['overall']['n_total'],
#         'accuracy'   : s['overall']['accuracy'],
#         'f1_macro'   : s['overall']['f1_macro'],
#         'f1_weighted': s['overall']['f1_weighted'],
#         'auc_macro'  : s['overall']['auc_macro'],
#     }
#     for cls_name, f1v in s['overall']['per_class_f1'].items():
#         safe = cls_name.replace(' ', '_').replace('/', '')
#         row[f'f1_{safe}'] = f1v
#     rows.append(row)

# comp_df = pd.DataFrame(rows).round(4)
# save_csv = 'Thesis_updated_results/BIRADS_results/all_models_comparison.csv'
# os.makedirs(os.path.dirname(save_csv), exist_ok=True)
# comp_df.to_csv(save_csv, index=False)
# print(f"\n{'='*70}\nCOMBINED RESULTS TABLE\n{'='*70}")
# print(comp_df.to_string(index=False))
# print(f"\nSaved → {save_csv}")